# RAG híbrido completo: pré/pós-processamento + NL2SQL + NL2Graph

Este notebook junta as três aulas:

### Primeira aula
- embeddings
- banco de dados vetoriais
- chunks
- overleap
- contexto

### Segunda aula
- ambiguidade;
- rewrite;
- multi-query;
- step-back;
- HyDE;
- múltiplas buscas;
- RRF;
- filtros;
- deduplicação;
- reranking;
- compressão;
- contexto suficiente.

### Aula atual
- roteamento;
- NL2SQL;
- execução controlada;
- NL2Graph;
- relações explícitas;
- evidências heterogêneas;
- permissões;
- versionamento;
- logs.

### Como estudar este notebook

1. Execute as seções na ordem numérica. Uma célula posterior utiliza variáveis e funções criadas nas células anteriores.
2. Leia primeiro o texto em Markdown; ele explica o objetivo antes do código correspondente.
3. Nos blocos de código, comentários iniciados por `#` descrevem o motivo de cada etapa. Não é necessário decorar tudo: observe o valor das variáveis impressas após cada execução.
4. Na primeira execução, o download do modelo e dos embeddings pode demorar. Depois, os arquivos ficam armazenados em cache.
5. Ao chegar à Atividade final, execute as células de teste e de geração do relatório. A célula de pergunta livre é opcional.

### Vocabulário mínimo

- **LLM**: modelo de linguagem que interpreta perguntas escritas em linguagem natural.
- **Evidência**: trecho de PDF, resultado SQL ou relação do grafo que justifica uma resposta.
- **Rota**: estratégia escolhida para recuperar a evidência. `vetor` busca por significado, `sql` busca valores exatos e `grafo` percorre relações.
- **Pipeline**: sequência organizada de etapas; neste caso, pergunta → recuperação → validação → resposta.

### Como ler o código, mesmo começando em Python

Os comentários no início de cada célula explicam o **objetivo**, as escolhas de implementação e o que observar. Junto às funções, explicam o que entra, o que sai e por que usamos aquela estratégia. Os comentários internos detalham as decisões de cada bloco.

- **`# comentário`** explica o código e não é executado. Texto entre aspas é um valor: pode ser uma pergunta, um prompt ou uma instrução SQL.
- **Indentação** (espaços à esquerda) delimita blocos de funções, condições e laços. Parênteses permitem quebrar uma instrução em várias linhas sem encerrá-la.
- **`def`** define uma função; `nome(...)` chama a função. Definir uma função não executa seu corpo. Variáveis criadas dentro dela normalmente só existem naquela chamada; objetos globais, como o modelo e o grafo, são reutilizados.
- **Lista `[...]`** mantém uma sequência; **tupla `(a, b)`** agrupa valores; **dicionário `{'chave': valor}`** associa nomes a dados; **conjunto `{'a', 'b'}`** reúne valores sem repetições. As chaves vazias `{}` criam um dicionário, não um conjunto.
- **`None`** representa ausência de valor. **`False`** é uma decisão falsa. Uma lista vazia também é considerada falsa dentro de um `if`, mas esses valores têm significados diferentes.
- **`registro['rota']`** exige que a chave exista; **`registro.get('rota', 'vetor')`** usa o padrão apenas quando ela não existe. Se a chave estiver presente com valor `None`, `.get()` retorna `None`.
- **`return`** encerra a função e devolve um resultado; **`raise`** sinaliza um erro; **`try/except`** permite tratar erros; **`finally`** executa a limpeza ao sair do bloco, mesmo se houver uma exceção.
- **`str`, `int`, `float` e `bool`** representam texto, inteiro, número com parte decimal e verdadeiro/falso. Anotações de tipo ajudam a leitura, mas não validam os dados automaticamente.

Para estudar uma função, acompanhe um exemplo: qual valor ela recebe, o que cada condição decide e qual valor retorna. Nos diagnósticos, compare os valores impressos com a intenção da pergunta. Um código que executou sem erro ainda pode produzir uma resposta inadequada.

A arquitetura final fica:

```text
                    PERGUNTA
                       ↓
                  ambiguidade
                       ↓
                  LLM roteador
                       ↓
┌───────────────────────────────────────────────┐
│                 ROTAS                         │
│                                               │
│ vetor       SQL        grafo      híbridas    │
│   ↓          ↓           ↓            ↓        │
│ pré       NL2SQL      NL2Graph     vetor +     │
│ vetor        ↓           ↓         SQL/grafo   │
│   ↓       validação   validação        ↓        │
│ busca      EXPLAIN      grafo      evidências  │
│   ↓       read-only                         │   │
│ RRF                                           │
│   ↓                                           │
│ pós-vetor                                     │
└───────────────────────────────────────────────┘
                       ↓
              evidências heterogêneas
                       ↓
              pós-processamento comum
                       ↓
                contexto suficiente?
                       ↓
                  LLM responde
                       ↓
                      log
```

> Ideia: **o LLM interpreta e propõe; o sistema valida e executa**.

## 1. Instalação

In [1]:
# Preparação: instalar bibliotecas no mesmo ambiente usado pelo Jupyter.
# O arquivo requirements.txt reúne as dependências para não instalar uma por vez.
# Estas linhas começam com #, portanto não executam comandos: copie o comando
# de instalação para o terminal. A biblioteca instalada em outro ambiente pode
# continuar indisponível para o kernel, que é o processo que executa as células.

# Ambiente local já definido em requirements.txt.
# Execute, no terminal, antes de abrir este notebook:
# .venv/bin/python -m pip install -r requirements.txt


## 2. Localização dos três PDFs no ambiente local

In [2]:
# Objetivo: localizar os PDFs que servirão como fontes das respostas.
# Path representa caminhos de arquivos e evita montar caminhos como texto manualmente.
# O operador / de Path junta pastas; aqui não é uma divisão matemática.
# mkdir(..., exist_ok=True) permite reexecutar a célula sem erro se a pasta já existir.
# glob("*.pdf") encontra arquivos com essa extensão e sorted dá uma ordem previsível.
# Esta célula apenas lista os PDFs: a leitura e a divisão do texto vêm depois.

from pathlib import Path

PASTA_RAIZ = Path.cwd()
PASTA_DOCUMENTOS = PASTA_RAIZ / "documentos"
PASTA_DOCUMENTOS.mkdir(parents=True, exist_ok=True)

print("Pasta de documentos:", PASTA_DOCUMENTOS)
print("\nArquivos disponíveis:")
arquivos_pdf = sorted(PASTA_DOCUMENTOS.glob("*.pdf"))
for arquivo in arquivos_pdf:
    print("-", arquivo.name)

# Uma lista vazia é considerada False; not inverte esse valor.
# Assim detectamos ausência de PDFs sem precisar escrever len(arquivos_pdf) == 0.
if not arquivos_pdf:
    print("Nenhum PDF encontrado. Adicione o corpus original em documentos/.")


Pasta de documentos: /home/ubuntu/mba-genai/CONT/aula05/documentos

Arquivos disponíveis:
- 2024-regulamento-estagio.pdf
- PPC-do-Curso-de-Ciencia-da-Computação.pdf
- ci1218.pdf


## 3. LLM local gratuito

Usaremos `Qwen/Qwen2.5-1.5B-Instruct` - LLM compacto e rápido, desenvolvido pela empresa chinesa Alibaba.

O LLM será usado para:
- detectar ambiguidade;
- gerar transformações da consulta;
- escolher a rota;
- gerar SQL;
- gerar consulta de grafo;
- extrair entidades;
- avaliar suficiência;
- redigir a resposta.


In [3]:
# Objetivo: preparar um modelo local que será reutilizado em várias funções.
# import disponibiliza bibliotecas; "as nx" cria um apelido para networkx.
# O tokenizer converte texto em números, e o LLM produz novos números que serão
# convertidos de volta em texto. Ambos devem pertencer ao mesmo modelo.
# O carregamento pode baixar arquivos na primeira execução e consumir bastante RAM.
# Carregamos o modelo uma vez, fora das funções, para não repetir esse custo a cada pergunta.

import re
import json
import time
import math
import sqlite3
import hashlib
import unicodedata
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import torch
import chromadb
import networkx as nx

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from sqlglot import parse, exp

MODELO_LLM = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODELO_LLM)

# A expressão A if condição else B escolhe um valor conforme a condição.
# Com CUDA usamos float16 para economizar memória; em CPU usamos float32.
# Essa escolha diz respeito à representação numérica dos pesos, não ao idioma do modelo.
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

llm = AutoModelForCausalLM.from_pretrained(
    MODELO_LLM,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)

if not torch.cuda.is_available():
    llm = llm.to("cpu")

# Ativa o modo de avaliação do modelo. Isso não treina nem faz uma pergunta.
# torch.no_grad, usado na geração, tem outra função: evitar cálculo de gradientes.
llm.eval()

print("LLM:", MODELO_LLM)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Mantido somente para a aula.
# Em produção, prefira logging estruturado em vez de print().
MODO_DEMO = True


# Objetivo: mostrar informações intermediárias quando MODO_DEMO estiver ativo.
# *args recebe vários argumentos posicionais e print(*args) os repassa separadamente.
# Uma chave global liga/desliga as mensagens sem alterar todas as chamadas.
def demo_log(*args):
    if MODO_DEMO:
        print(*args)

/home/ubuntu/mba-genai/CONT/aula05/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


LLM: Qwen/Qwen2.5-1.5B-Instruct
CUDA: True
GPU: NVIDIA RTX 2000 Ada Generation


In [4]:
# Objetivo: oferecer uma interface única para conversar com o LLM e ler sua saída.
# def define uma função; o corpo indentado só roda quando a função é chamada.
# Os nomes entre parênteses são parâmetros. "max_new_tokens=250" define um padrão
# que a chamada pode substituir. ": str" e "-> str" documentam tipos esperados;
# essas anotações não convertem valores nem validam tipos automaticamente.
# return devolve um valor ao código que chamou a função; print apenas o exibe.

# Centraliza a conversa com o modelo para que as demais células não precisem
# repetir detalhes técnicos de tokenizer, GPU/CPU e geração de texto.
# Recebe instruções, conteúdo e limite de geração; devolve somente a resposta em texto.
# Centralizar a chamada mantém o mesmo formato de conversa e tratamento de dispositivo.
# max_new_tokens limita a continuação gerada, não o comprimento total do prompt.
def chamar_llm(
    system: str,
    user: str,
    max_new_tokens: int = 250,
) -> str:
    # O modelo recebe uma instrução geral (system) e a pergunta/dado do usuário.
    mensagens = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

    # Converte a lista de mensagens para o formato de conversa esperado pelo modelo.
    prompt = tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True,
    )

    # return_tensors="pt" pede tensores do PyTorch: arrays numéricos que o modelo processa.
    # O dicionário inclui IDs dos tokens e outras informações, como a máscara de atenção.
    entradas = tokenizer(
        prompt,
        return_tensors="pt",
    )

    # Descobrimos automaticamente se o modelo está em CPU ou GPU e movemos
    # os dados de entrada para o mesmo lugar.
    device = next(llm.parameters()).device
    entradas = {
        chave: valor.to(device)
        for chave, valor in entradas.items()
    }

    # Não calculamos gradientes porque estamos apenas consultando o modelo,
    # não treinando-o. Isso economiza memória.
    with torch.no_grad():
        # **entradas desempacota o dicionário como argumentos nomeados de generate.
        # do_sample=False evita sortear tokens; isso reduz variação entre execuções,
        # mas não transforma a saída do modelo em uma resposta factual garantida.
        saida = llm.generate(
            **entradas,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            # O modelo pode trazer parâmetros de amostragem no arquivo de configuração.
            # Como não sorteamos tokens, usamos os padrões aceitos pelo Transformers
            # para evitar avisos de opções incompatíveis com do_sample=False.
            temperature=1.0,
            top_p=1.0,
            top_k=50,
            pad_token_id=tokenizer.eos_token_id,
        )

    # A saída contém o prompt original e a continuação. Mantemos apenas
    # os novos tokens: isto é, a resposta criada pelo modelo.
    novos = saida[0][entradas["input_ids"].shape[1]:]

    return tokenizer.decode(
        novos,
        skip_special_tokens=True,
    ).strip()


# Remove cercas de Markdown que o modelo pode acrescentar ao SQL ou JSON.
# Usamos expressões regulares nas extremidades para preservar o conteúdo interno.
# Isso é limpeza de apresentação; não valida a consulta ou o objeto retornado.
def limpar_bloco(texto: str) -> str:
    texto = texto.strip()
    texto = re.sub(
        r"^```(?:sql|json)?\s*",
        "",
        texto,
        flags=re.I,
    )
    texto = re.sub(r"\s*```$", "", texto)
    return texto.strip()


# Converte a parte entre a primeira { e a última } em um dicionário Python.
# O JSON permite acessar campos pelo nome em vez de interpretar frases livres.
# Se a estrutura não for válida, uma exceção informa o problema ao chamador.
# A estratégia pressupõe um único objeto; múltiplos objetos podem causar erro.
def extrair_json(texto: str) -> dict:
    texto = limpar_bloco(texto)

    inicio = texto.find("{")
    fim = texto.rfind("}")

    if inicio < 0 or fim < inicio:
        raise ValueError(
            f"JSON não encontrado na saída do LLM:\n{texto}"
        )

    return json.loads(
        texto[inicio:fim + 1]
    )

# Traduz variantes como 'false' e 'não' para os booleanos False/True.
# bool('false') seria True porque a string não está vazia; daí a comparação explícita.
# Testamos bool antes de números, pois bool também é uma subclasse de int em Python.
def normalizar_bool(valor, padrao=False) -> bool:
    """
    Normaliza booleanos retornados pelo LLM.

    Alguns modelos podem retornar:
      false
      "false"
      "False"
      0
      "não"

    Sem esta normalização, a string "false" é True em Python
    porque qualquer string não vazia é truthy (valor verdadeiro).
    """
    if isinstance(valor, bool):
        return valor

    if valor is None:
        return padrao

    if isinstance(valor, (int, float)):
        return bool(valor)

    texto = str(valor).strip().lower()

    verdadeiros = {
        "true", "1", "sim", "yes", "verdadeiro"
    }

    falsos = {
        "false", "0", "nao", "não", "no", "falso",
        "none", "null", ""
    }

    if texto in verdadeiros:
        return True

    if texto in falsos:
        return False

    return padrao

# PARTE I — Base vetorial

## 4. Leitura, metadados e chunking

Os metadados serão importantes mais tarde para:
- filtro por escopo;
- permissão;
- versionamento;
- rastreabilidade.

In [5]:
# Objetivo: transformar páginas de PDFs em pequenos registros pesquisáveis.
# Cada registro associa texto a arquivo, página, escopo e versão. Assim uma resposta
# pode indicar sua origem, e a busca pode filtrar documentos antes de responder.
# Embeddings são vetores numéricos que permitem comparar textos por proximidade.
# A leitura abaixo usa texto extraível do PDF; ela não faz OCR de páginas escaneadas.
# Versões e permissões são configurações didáticas definidas neste notebook.

MODELO_EMBEDDING = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

modelo_embedding = SentenceTransformer(
    MODELO_EMBEDDING
)

PASTA_CHROMA = PASTA_RAIZ / "chroma"
NOME_COLECAO = "documentos_academicos_hibrido"

# Estes números contam caracteres, não palavras ou tokens.
# OVERLAP deve ser menor que TAMANHO_CHUNK para o passo de geração ser positivo.
TAMANHO_CHUNK = 800
OVERLAP = 150

# Um dicionário permite buscar a versão pelo escopo: VERSOES_ATUAIS["estagio"].
# São rótulos do corpus da aula, não uma consulta automática à norma mais recente.
VERSOES_ATUAIS = {
    "curso": "2018",
    "estagio": "2024",
    "disciplina": "ficha_ci1218",
}

PERMISSOES = {
    "publico": {"curso", "estagio", "disciplina", "geral"},
    "somente_curso": {"curso"},
    "somente_estagio": {"estagio"},
    "somente_disciplina": {"disciplina"},
}


# Produz uma versão sem acentos para comparar palavras com grafias diferentes.
# NFD separa a letra de sua marca de acento; o filtro remove essas marcas.
# O texto original do PDF é mantido nos registros para citação posterior.
def remover_acentos(texto: str) -> str:
    decomposto = unicodedata.normalize(
        "NFD",
        texto,
    )

    return "".join(
        char
        for char in decomposto
        if unicodedata.category(char) != "Mn"
    )


# Padroniza maiúsculas, acentos e separadores para as regras de comparação.
# split seguido de join reduz sequências de espaços a um único espaço.
# Essa representação serve às buscas por termos, não à exibição do documento original.
def normalizar(texto: str) -> str:
    texto = remover_acentos(
        texto.lower()
    )
    texto = re.sub(
        r"[^a-z0-9]+",
        " ",
        texto,
    )
    return " ".join(
        texto.split()
    )


# Classifica o registro como curso, estágio, disciplina ou geral.
# Usa nome do arquivo e início do texto por ser uma regra curta para este corpus.
# A ordem dos testes importa quando há mais de um indício; esta é uma heurística,
# não uma classificação infalível nem uma leitura da estrutura completa do PDF.
def inferir_escopo(
    nome: str,
    texto: str = "",
) -> str:
    nome_norm = normalizar(nome)
    alvo = normalizar(
        nome + " " + texto[:1200]
    )

    if "estagio" in nome_norm:
        return "estagio"

    if (
        "ci1218" in alvo
        or "ficha" in alvo
        or "ementa" in alvo
    ):
        return "disciplina"

    if (
        "ppc" in alvo
        or "projeto pedagogico" in alvo
    ):
        return "curso"

    return "geral"


# Obtém o rótulo de versão configurado para o escopo inferido.
# .get(chave, padrão) devolve o padrão se a chave não estiver no dicionário.
# Aqui a versão vem da configuração: não é extraída de datas dentro do PDF.
def inferir_versao(
    nome: str,
    texto: str = "",
) -> str:
    escopo = inferir_escopo(
        nome,
        texto,
    )

    return VERSOES_ATUAIS.get(
        escopo,
        "desconhecida",
    )


# Divide o texto em janelas de caracteres para criar unidades pequenas de busca.
# Com tamanho 800 e overlap 150, o passo é 650: o próximo trecho repete 150 caracteres.
# A repetição ajuda a preservar contexto na fronteira, mas pode gerar duplicatas.
# O corte é por caracteres e pode atravessar palavras; não é uma divisão por sentenças.
def gerar_chunks(texto: str) -> list[str]:
    saida = []
    passo = TAMANHO_CHUNK - OVERLAP

    for inicio in range(
        0,
        len(texto),
        passo,
    ):
        trecho = texto[
            inicio:inicio + TAMANHO_CHUNK
        ].strip()

        if trecho:
            saida.append(trecho)

    return saida


# Lê cada página e devolve uma lista de dicionários, um por trecho.
# O ID combina arquivo, página e número do trecho para permitir rastreamento.
# enumerate(..., start=1) usa numeração humana de páginas, começando em 1.
# Páginas sem texto extraível são ignoradas; nenhum OCR é executado aqui.
def carregar_corpus() -> list[dict]:
    registros = []

    # genia_3.pdf é o slide da aula, não uma fonte do corpus acadêmico.
    for arquivo in sorted(
        arquivo
        for arquivo in PASTA_DOCUMENTOS.glob("*.pdf")
        if arquivo.name != "genia_3.pdf"
    ):
        reader = PdfReader(
            str(arquivo)
        )

        for pagina_num, pagina in enumerate(
            reader.pages,
            start=1,
        ):
            # extract_text pode retornar None; "or ''" oferece uma string vazia nesse caso.
            # strip remove espaços nas extremidades para detectar páginas sem conteúdo útil.
            texto = (
                pagina.extract_text() or ""
            ).strip()

            if not texto:
                continue

            escopo = inferir_escopo(
                arquivo.name,
                texto,
            )

            versao = inferir_versao(
                arquivo.name,
                texto,
            )

            for chunk_num, trecho in enumerate(
                gerar_chunks(texto),
                start=1,
            ):
                # append adiciona um registro por trecho. O dicionário mantém dados e origem juntos.
                # Na f-string, :03d mostra inteiros com três dígitos, como 001, facilitando ler os IDs.
                registros.append({
                    "id": (
                        f"{arquivo.stem}"
                        f"_p{pagina_num:03d}"
                        f"_c{chunk_num:03d}"
                    ),
                    "texto": trecho,
                    "arquivo": arquivo.name,
                    "pagina": pagina_num,
                    "chunk": chunk_num,
                    "escopo": escopo,
                    "versao": versao,
                    "embedding_version": versao,
                })

    return registros


registros = carregar_corpus()

print("Chunks:", len(registros))
print(
    "Escopos:",
    sorted(
        {item["escopo"] for item in registros}
    ),
)

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 64 0 (offset 0)
Ignoring wrong pointing object 66 0 (offset 0)
Ignoring wrong pointing object 92 0 (offset 0)
Ignoring wrong pointing object 98 0 (offset 0)
Ignoring wrong pointing object 172 0 (offset 0)
Ignoring wrong pointing object 9 0 (offset 0)


Chunks: 128
Escopos: ['curso', 'disciplina', 'estagio']


## 5. Indexação em ChromaDB

In [6]:
# Objetivo: armazenar os trechos e seus embeddings em uma coleção pesquisável.
# PersistentClient grava o índice na pasta chroma, em vez de mantê-lo só em memória.
# Esta célula remove e recria a coleção indicada para reindexar o corpus da aula.
# Essa escolha evita reutilizar trechos antigos, mas repetir a célula refaz o trabalho.
# Os metadados viajam junto com cada vetor para preservar página, versão e escopo.

PASTA_CHROMA.mkdir(
    parents=True,
    exist_ok=True,
)

cliente = chromadb.PersistentClient(
    path=str(PASTA_CHROMA)
)

# Tentamos remover a coleção anterior para recomeçar a indexação.
# O except abaixo ignora qualquer Exception, não apenas 'coleção inexistente';
# portanto ele pode ocultar a causa original se a remoção falhar por outro motivo.
try:
    cliente.delete_collection(
        NOME_COLECAO
    )
except Exception:
    pass

colecao = cliente.get_or_create_collection(
    name=NOME_COLECAO,
    metadata={
        "hnsw:space": "cosine"
    },
)

# Processar 32 trechos de cada vez limita o volume enviado ao modelo por chamada.
# O for avança de LOTE em LOTE; a fatia final pode ter menos de 32 itens.
LOTE = 32

for inicio in range(
    0,
    len(registros),
    LOTE,
):
    lote = registros[
        inicio:inicio + LOTE
    ]

    # Compreensão de lista: leia como 'pegue item["texto"] para cada item do lote'.
    # Produzimos uma lista simples de strings porque o modelo não recebe metadados.
    textos = [
        item["texto"]
        for item in lote
    ]

    # normalize_embeddings=True ajusta os vetores para comprimento unitário.
    # Isso combina com a distância de cosseno usada na coleção para comparar direção.
    embeddings = modelo_embedding.encode(
        textos,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    # IDs, textos, vetores e metadados precisam seguir a mesma ordem.
    # embeddings.tolist() converte o array numérico no formato de listas da chamada.
    colecao.add(
        ids=[
            item["id"]
            for item in lote
        ],
        documents=textos,
        embeddings=embeddings.tolist(),
        metadatas=[
            {
                "arquivo": item["arquivo"],
                "pagina": item["pagina"],
                "chunk": item["chunk"],
                "escopo": item["escopo"],
                "versao": item["versao"],
                "embedding_version": item[
                    "embedding_version"
                ],
            }
            for item in lote
        ],
    )

print(
    "Registros na coleção:",
    colecao.count(),
)

Batches: 100%|██████████| 1/1 [00:00<00:00, 22.28it/s]


Registros na coleção: 128


# PARTE II — Pré-processamento

## 6. Primeiro: ambiguidade

Ambiguidade é tratada **antes da recuperação**.

Exemplo:
> “Qual é a carga horária?”

Carga horária de quê?

Em vez de recuperar qualquer coisa, o sistema pode pedir esclarecimento.

In [7]:
# Objetivo: decidir se falta uma informação essencial antes de fazer uma busca.
# Usamos regras simples primeiro porque muitas perguntas já indicam claramente o alvo.
# O LLM é consultado apenas quando a pergunta combina com padrões vagos cadastrados.
# O resultado é um dicionário: pares chave/valor, como {"ambigua": False}.
# O chamador consulta essas chaves para decidir entre continuar e pedir esclarecimento.
# Essas regras são heurísticas da aula: reconhecer uma palavra não prova ausência de ambiguidade.

# Devolve uma decisão estruturada antes de buscar dados.
# Retornos antecipados evitam chamar o LLM quando uma regra já decidiu.
# Se o avaliador falhar, esta implementação permite seguir e registra o erro;
# esse comportamento prioriza continuidade e pode deixar passar perguntas vagas.
def detectar_ambiguidade_llm(
    pergunta: str,
) -> dict:
    """
    Detector de ambiguidade de ALTA PRECISÃO.

    Regra didática:
    - se a pergunta já identifica claramente o alvo, NÃO interromper;
    - só perguntar ao LLM quando a pergunta parece realmente subespecificada.

    Isso evita que o sistema transforme qualquer pergunta curta em
    "preciso esclarecer".
    """

    p = normalizar(pergunta)

    # Alvos explícitos: se qualquer um estiver presente, a pergunta
    # normalmente já tem escopo suficiente para seguir ao roteamento.
    alvos_explicitos = [
        "ci1218",
        "bancos de dados",
        "tcc",
        "tccs",
        "estagio obrigatorio",
        "estagio nao obrigatorio",
        "regulamento de estagio",
        "ppc",
        "projeto pedagogico",
        "atividades formativas",
        "pet",
        "curso de ciencia da computacao",
        "disciplina que trata",
        "pre requisitos",
        "prerequisitos",
        "pre requisito",
    ]

    # any retorna True se ao menos uma verificação for verdadeira.
    # O gerador testa presença de cada expressão na pergunta normalizada.
    # Esse atalho economiza chamadas, mas usa correspondência textual simplificada.
    if any(alvo in p for alvo in alvos_explicitos):
        return {
            "ambigua": False,
            "motivo": "a pergunta já contém um alvo explícito",
            "pergunta_esclarecimento": None,
        }

    # Só chamamos o LLM para padrões realmente vagos.
    gatilhos_vagos = [
        "qual e a carga horaria",
        "qual a carga horaria",
        "qual e a frequencia",
        "qual a frequencia",
        "qual e a regra",
        "qual a regra",
        "quais sao as regras",
        "qual documento",
        "qual e o documento",
    ]

    parece_vaga = any(gatilho in p for gatilho in gatilhos_vagos)

    if not parece_vaga:
        return {
            "ambigua": False,
            "motivo": "não há sinal forte de ambiguidade crítica",
            "pergunta_esclarecimento": None,
        }

    system = """
Decida se a pergunta está AMBÍGUA A PONTO DE IMPEDIR A EXECUÇÃO.

Se houver uma interpretação natural e útil, marque ambigua=false.
Use ambigua=true SOMENTE se faltar uma informação essencial para escolher
o alvo da consulta.

Exemplos:
"Qual é a carga horária?" -> ambigua=true
"Qual é a carga horária da CI1218?" -> ambigua=false
"Qual é a carga horária total dos TCCs?" -> ambigua=false
"Quais são os pré-requisitos da CI1218?" -> ambigua=false
"O que Bancos de Dados aborda sobre transações?" -> ambigua=false

Retorne APENAS JSON:
{
  "ambigua": false,
  "motivo": "...",
  "pergunta_esclarecimento": null
}

Não responda à pergunta do usuário.
""".strip()

    # Interpretamos a decisão de ambiguidade produzida pelo LLM.
    try:
        resultado = extrair_json(
            chamar_llm(
                system,
                pergunta,
                max_new_tokens=130,
            )
        )

        # CRÍTICO: normaliza "false" string para False boolean.
        resultado["ambigua"] = normalizar_bool(
            resultado.get("ambigua"),
            padrao=False,
        )

        if not resultado["ambigua"]:
            resultado["pergunta_esclarecimento"] = None

        return resultado

    except Exception as erro:
        # a pergunta segue ao roteamento em vez de travar.
        return {
            "ambigua": False,
            "motivo": (
                "falha na avaliação de ambiguidade; "
                "a pergunta seguirá ao roteamento"
            ),
            "pergunta_esclarecimento": None,
            "erro_detector": str(erro),
        }

## 7. Rewrite, Multi-query, Step-back e HyDE

Essas transformações serão aplicadas **somente quando a rota usar vetor**.

Isso é importante: não faz sentido aplicar HyDE a uma pergunta que será resolvida diretamente por SQL.

Em uma única chamada ao LLM produzimos:

- `rewrite`: versão mais clara;
- `expansoes`: perspectivas alternativas;
- `step_back`: pergunta mais geral;
- `hyde`: pequeno texto hipotético usado **somente para recuperação**.

> HyDE nunca é evidência.

In [8]:
# Objetivo: criar variações da pergunta para aumentar as chances de achar bons trechos.
# Perguntas equivalentes podem usar palavras diferentes das encontradas no documento.
# Por isso buscamos a formulação original e variações, mas limitamos a expansão
# para controlar tempo e evitar desviar do assunto.
# O texto hipotético de HyDE serve para buscar; não é uma fonte factual da resposta.

# Gera reformulações com a mesma intenção para ampliar a cobertura da busca.
# Limitamos expansões a duas para conter o número de consultas.
# Se a saída do modelo falhar, reutilizamos a pergunta original: a busca continua,
# mas aquela tentativa não se beneficia das variações planejadas.
def gerar_transformacoes_llm(
    pergunta: str,
) -> dict:
    """
    Gera transformações CONSERVADORAS para recuperação.

    O modelo não deve acrescentar tópicos que não aparecem
    na intenção original. Isso reduz query drift.
    """

    system = """
Você prepara consultas para recuperação vetorial.

Retorne APENAS JSON:
{
  "rewrite": "...",
  "expansoes": ["...", "..."],
  "step_back": "...",
  "hyde": "..."
}

REGRAS IMPORTANTES:
- preserve rigorosamente a intenção original;
- NÃO acrescente novos assuntos, propriedades ou requisitos;
- rewrite deve apenas tornar a pergunta mais explícita;
- expansoes devem ser paráfrases curtas da mesma necessidade informacional;
- step_back deve ser uma formulação um pouco mais geral,
  sem mudar de domínio;
- hyde deve ser UMA frase hipotética curta que use os conceitos
  centrais da pergunta;
- HyDE é somente consulta de recuperação e nunca evidência;
- escreva em português.


Não invente temas como segurança, indexação ou sistemas distribuídos
se eles não estiverem explicitamente na pergunta.
""".strip()

    # Guardamos a saída antes de tentar lê-la como JSON. Assim um erro de
    # formatação não apaga a informação necessária para investigar a causa.
    saida_bruta = None
    try:
        saida_bruta = chamar_llm(
            system,
            pergunta,
            max_new_tokens=240,
        )
        resultado = extrair_json(saida_bruta)

        # JSON válido ainda pode ter campos ausentes ou tipos inadequados.
        # Validamos a estrutura antes de usar .strip() nas consultas.
        for campo in ("rewrite", "step_back", "hyde"):
            if not isinstance(resultado.get(campo), str) or not resultado[campo].strip():
                raise ValueError(f"Campo {campo!r} deve conter texto não vazio.")

        expansoes = resultado.get("expansoes", [])

        # Uma lista vazia é permitida; uma string ou itens numéricos não são consultas.
        if not isinstance(expansoes, list) or any(not isinstance(item, str) for item in expansoes):
            raise ValueError("expansoes deve ser uma lista de textos.")

        return {
            "rewrite": resultado.get("rewrite") or pergunta,
            "expansoes": expansoes[:2],
            "step_back": resultado.get("step_back") or pergunta,
            "hyde": resultado.get("hyde") or pergunta,
            "diagnostico": {"origem": "llm", "erro": None},
        }

    except Exception as erro:
        # Se o LLM falhar, o sistema não para: usa a pergunta original
        # para continuar a recuperação. O diagnóstico distingue essa alternativa
        # de uma geração válida que apenas repetiu a pergunta.
        return {
            "rewrite": pergunta,
            "expansoes": [],
            "step_back": pergunta,
            "hyde": pergunta,
            "diagnostico": {
                "origem": "fallback_pergunta_original",
                "erro": f"{type(erro).__name__}: {erro}",
                "saida_bruta": saida_bruta,
            },
        }


# Junta a pergunta original às reformulações e remove vazios/repetições.
# *lista espalha seus elementos na lista externa; sem * teríamos uma lista dentro de outra.
# dict.fromkeys elimina repetidos preservando a ordem da primeira ocorrência.
# Devolver também transformacoes permite inspecionar o que o LLM sugeriu.
def montar_consultas_vetoriais(
    pergunta: str,
) -> dict:
    transformacoes = gerar_transformacoes_llm(pergunta)

    consultas = [
        pergunta,  # a pergunta original SEMPRE permanece
        transformacoes["rewrite"],
        *transformacoes["expansoes"],
        transformacoes["step_back"],
        transformacoes["hyde"],
    ]

    consultas = [
        consulta.strip()
        for consulta in consultas
        if consulta and consulta.strip()
    ]

    consultas = list(dict.fromkeys(consultas))

    # Contamos consultas distintas: várias transformações iguais viram uma só.
    # Esse diagnóstico descreve o que ocorreu sem afirmar que a busca melhorou.
    transformacoes["diagnostico"]["quantidade_consultas_distintas"] = len(consultas)
    transformacoes["diagnostico"]["houve_variacao"] = any(
        consulta != pergunta.strip() for consulta in consultas
    )

    return {
        "transformacoes": transformacoes,
        "consultas": consultas,
    }

### Veja o pré-processamento sem executar a busca

Observe `transformacoes.diagnostico` no resultado:

- `origem = llm`: a saída foi interpretada e passou nas verificações de estrutura. Isso não garante que houve reformulação útil.
- `origem = fallback_pergunta_original`: alguma etapa falhou e a busca continuará com a pergunta original. Leia `erro` e `saida_bruta` para investigar.
- `houve_variacao = false`: nenhuma consulta distinta da pergunta foi produzida.
- `quantidade_consultas_distintas`: número de consultas que serão executadas após remover repetições.

Um JSON incompleto pode indicar corte pelo limite de tokens, mas confirme pela saída bruta antes de alterar esse limite. O aviso sobre parâmetros de amostragem, isoladamente, não explica uma falha na leitura do JSON. Após editar funções no notebook, reexecute suas células para atualizar as definições em memória.

In [9]:
# Objetivo: inspecionar como a pergunta foi reformulada antes de buscar documentos.
# As duas strings entre parênteses são unidas automaticamente pelo Python.
# pre recebe um dicionário com transformações e consultas geradas.
# json.dumps converte esse dicionário em texto; indent=2 organiza a visualização
# e ensure_ascii=False mantém os acentos legíveis. Confira se as novas consultas
# preservam a intenção original; esta célula ainda não responde à pergunta.

pergunta = (
    "O que Bancos de Dados aborda "
    "sobre processamento de consultas?"
)

pre = montar_consultas_vetoriais(
    pergunta
)

print(
    json.dumps(
        pre,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "transformacoes": {
    "rewrite": "Como o Banco de Dados trata as consultas?",
    "expansoes": [
      "Banco de Dados: Processamento de Consultas",
      "Consultas no Banco de Dados"
    ],
    "step_back": "Processamento de dados",
    "hyde": "Banco de Dados: Consultas fundamentais",
    "diagnostico": {
      "origem": "llm",
      "erro": null,
      "quantidade_consultas_distintas": 6,
      "houve_variacao": true
    }
  },
  "consultas": [
    "O que Bancos de Dados aborda sobre processamento de consultas?",
    "Como o Banco de Dados trata as consultas?",
    "Banco de Dados: Processamento de Consultas",
    "Consultas no Banco de Dados",
    "Processamento de dados",
    "Banco de Dados: Consultas fundamentais"
  ]
}


# PARTE III — Recuperação vetorial avançada

## 8. Busca múltipla

Cada variante gerada no pré-processamento realiza uma busca.

Depois combinaremos os rankings usando **RRF — Reciprocal Rank Fusion**.

In [10]:
# Objetivo: padronizar resultados de busca e aplicar os filtros de acesso.
# @dataclass cria automaticamente a inicialização de objetos com os campos declarados.
# ResultadoVetorial guarda um candidato bruto; Evidencia representa o material
# preparado para justificar a resposta, seja ele vetorial, SQL ou de grafo.
# Guardar fonte e metadados no próprio objeto permite rastrear cada etapa.
# A filtragem do Chroma ocorre na consulta, antes de os trechos serem retornados.

@dataclass
class ResultadoVetorial:
    id: str
    texto: str
    fonte: str
    metadata: dict
    similaridade: float
    consulta_origem: str
    rank_origem: int
    rrf_score: float = 0.0


@dataclass
class Evidencia:
    tipo: str
    conteudo: str
    fonte: str
    metadata: dict



# Obtém o conjunto de categorias que um perfil pode consultar.
# set facilita verificações de pertencimento com in e ignora repetições.
# Um perfil desconhecido gera PermissionError em vez de receber acesso padrão.
def escopos_permitidos(
    perfil: str,
) -> set[str]:
    if perfil not in PERMISSOES:
        raise PermissionError(
            f"Perfil desconhecido: {perfil}"
        )

    return set(
        PERMISSOES[perfil]
    )


# Recusa antecipadamente um escopo explicitamente fora do perfil.
# Retorno -> None significa que a função valida ou lança erro, sem devolver dados.
# Se escopo for None, este teste não escolhe uma categoria; o filtro da busca
# vetorial ainda precisa restringir os documentos aos escopos autorizados.
def validar_permissao(
    perfil: str,
    escopo: str | None,
) -> None:
    """
    Validação antecipada.

    IMPORTANTE:
    esta função não é mais a única barreira.
    A recuperação aplica ACL novamente no nível dos documentos.
    """
    permitidos = escopos_permitidos(
        perfil
    )

    if (
        escopo is not None
        and escopo not in permitidos
    ):
        raise PermissionError(
            f"Perfil '{perfil}' não possui "
            f"acesso ao escopo '{escopo}'."
        )


# Converte a política de perfis no formato de filtro aceito pela consulta.
# Um escopo gera igualdade; vários escopos geram $in, isto é, 'um destes valores'.
# Aplicar o filtro na busca evita recuperar documentos fora das categorias permitidas.
def filtro_chroma_por_permissao(
    perfil: str,
    escopo: str | None = None,
) -> dict:
    """
    ACL no nível da recuperação.

    Mesmo se o roteador devolver escopo=None, a consulta
    só enxerga documentos pertencentes aos escopos permitidos.
    """
    permitidos = escopos_permitidos(
        perfil
    )

    if escopo is not None:
        validar_permissao(
            perfil,
            escopo,
        )
        return {
            "escopo": escopo
        }

    permitidos_ordenados = sorted(
        permitidos
    )

    if len(permitidos_ordenados) == 1:
        return {
            "escopo": permitidos_ordenados[0]
        }

    return {
        "escopo": {
            "$in": permitidos_ordenados
        }
    }


# Transforma uma consulta em embedding e recupera até k candidatos do Chroma.
# Usamos o mesmo modelo de embeddings da indexação para comparar vetores compatíveis.
# A saída mantém texto, fonte, metadados e posição, que serão usados no pós-processamento.
def buscar_uma_consulta(
    consulta: str,
    escopo: str | None = None,
    k: int = 5,
    perfil: str = "publico",
) -> list[ResultadoVetorial]:
    embedding = modelo_embedding.encode(
        consulta,
        normalize_embeddings=True,
    ).tolist()

    params = {
        "query_embeddings": [embedding],
        "n_results": k,
        "include": [
            "documents",
            "metadatas",
            "distances",
        ],
        # A ACL não depende do acerto do roteador.
        "where": filtro_chroma_por_permissao(
            perfil,
            escopo,
        ),
    }

    # params é um dicionário de opções. **params equivale a passar cada chave
    # como argumento nomeado: query_embeddings=..., n_results=..., where=....
    resultado = colecao.query(
        **params
    )

    saida = []

    # O Chroma devolve uma lista de resultados por consulta; [0] escolhe a única consulta.
    # zip alinha ID, texto, metadados e distância da mesma posição.
    # enumerate acrescenta a posição no ranking, começando em 1.
    for rank, (
        identificador,
        documento,
        metadata,
        distancia,
    ) in enumerate(
        zip(
            resultado["ids"][0],
            resultado["documents"][0],
            resultado["metadatas"][0],
            resultado["distances"][0],
        ),
        start=1,
    ):
        saida.append(
            ResultadoVetorial(
                id=identificador,
                texto=documento,
                fonte=(
                    f"{metadata['arquivo']} "
                    f"p.{metadata['pagina']}"
                ),
                metadata=metadata,
                # Com distância de cosseno, 1 - distância expressa uma medida de similaridade.
                # Maior tende a indicar maior proximidade; o valor não é probabilidade de acerto.
                similaridade=(
                    1 - float(distancia)
                ),
                consulta_origem=consulta,
                rank_origem=rank,
            )
        )

    return saida

## 9. RRF — fusão dos rankings

In [11]:
# Objetivo: combinar resultados de várias consultas sem duplicar o mesmo trecho.
# Cada consulta gera seu próprio ranking; RRF soma contribuições baseadas na posição.
# Isso permite combinar rankings sem tratar seus scores brutos como equivalentes.
# Dois dicionários usam o ID do trecho como chave: um acumula pontos e o outro
# guarda o objeto representativo. A função de recuperação retorna dois valores:
# os candidatos ordenados e o registro das transformações feitas na pergunta.

# RRF (Reciprocal Rank Fusion) combina rankings de várias consultas.
# Um documento bem posicionado em mais de uma busca recebe pontuação maior.
# Soma contribuições 1/(k_rrf + posição) para cada ID presente nos rankings.
# Com k_rrf=60, posições próximas contribuem de modo parecido; não são ignoradas.
# O dicionário acumula scores entre consultas e evita devolver um objeto por repetição.
def fusao_rrf(
    listas: list[list[ResultadoVetorial]],
    k_rrf: int = 60,
) -> list[ResultadoVetorial]:
    # `scores` acumula a pontuação final; `melhores` guarda o melhor trecho
    # encontrado para cada identificador de documento.
    scores = {}
    melhores = {}

    # Percorremos os resultados de todas as reformulações da pergunta.
    for lista in listas:
        for posicao, item in enumerate(
            lista,
            start=1,
        ):
            # get(id, 0.0) inicia a soma em zero quando esse ID aparece pela primeira vez.
            # Cada ocorrência contribui conforme a posição; o mesmo trecho pode somar pontos
            # de várias reformulações da pergunta.
            scores[item.id] = (
                scores.get(
                    item.id,
                    0.0,
                )
                + 1.0 / (
                    k_rrf + posicao
                )
            )

            if (
                item.id not in melhores
                or item.similaridade
                > melhores[item.id].similaridade
            ):
                melhores[item.id] = item

    saida = []

    # Por fim, ordenamos do maior para o menor score para produzir um ranking.
    # key=lambda par: par[1] escolhe a segunda posição do par (ID, score) para ordenar.
    # lambda cria uma função curta; reverse=True coloca as maiores pontuações primeiro.
    for identificador, score in sorted(
        scores.items(),
        key=lambda par: par[1],
        reverse=True,
    ):
        item = melhores[
            identificador
        ]
        item.rrf_score = score
        saida.append(item)

    return saida


# Executa cada formulação e combina seus candidatos por RRF.
# A lista de rankings preserva a posição local antes da fusão.
# return candidatos, pre devolve uma tupla de dois valores para o chamador desempacotar.
def recuperar_multi_query(
    pergunta: str,
    escopo: None, #str | None = None
    k_por_consulta: int = 5,
    perfil: str = "publico",
) -> tuple[list[ResultadoVetorial], dict]:
    pre = montar_consultas_vetoriais(
        pergunta
    )

    listas = []

    for consulta in pre["consultas"]:
        listas.append(
            buscar_uma_consulta(
                consulta,
                escopo=escopo,
                k=k_por_consulta,
                perfil=perfil,
            )
        )

    candidatos = fusao_rrf(
        listas
    )

    return candidatos, pre

# PARTE IV — Pós-processamento vetorial

## 10. Filtro por score e versionamento

Agora tratamos os resultados recuperados.

Primeiro:
- eliminamos similaridades muito fracas;
- eliminamos embeddings cuja versão não corresponde mais à versão atual do documento.

Esse segundo passo permite discutir **invalidação de embeddings**.

In [12]:
# Objetivo: retirar candidatos pouco parecidos ou marcados com uma versão antiga.
# O limiar 0.20 é um parâmetro didático de seleção, não uma probabilidade de acerto.
# A versão vem dos metadados atribuídos ao carregar o corpus; este filtro compara
# esses rótulos e não verifica sozinho se um documento continua vigente.
# Separar os dois filtros permite investigar qual deles eliminou um candidato.

LIMIAR_SIMILARIDADE = 0.20


# Devolve somente itens cuja similaridade atende ao mínimo escolhido.
# A compreensão de lista equivale a um for com if e append, criando uma nova lista.
# minimo recebe o padrão da definição; para testar outro limiar, passe-o na chamada.
def filtrar_score(
    candidatos: list[ResultadoVetorial],
    minimo: float = LIMIAR_SIMILARIDADE,
) -> list[ResultadoVetorial]:
    return [
        item
        for item in candidatos
        if item.similaridade >= minimo
    ]


# Mantém itens com embedding_version igual ao rótulo configurado para seu escopo.
# Se não existe versão configurada, esta implementação também mantém o item.
# Separar a regra permite demonstrar por que um resultado semelhante pode ser descartado.
def filtrar_versao(
    candidatos: list[ResultadoVetorial],
) -> list[ResultadoVetorial]:
    saida = []

    for item in candidatos:
        escopo = item.metadata.get(
            "escopo"
        )

        versao_atual = (
            VERSOES_ATUAIS.get(
                escopo
            )
        )

        embedding_version = (
            item.metadata.get(
                "embedding_version"
            )
        )

        # A condição tem duas alternativas ligadas por or: não há versão configurada
        # OU o item coincide com ela. Logo, falta de configuração não elimina o trecho.
        if (
            versao_atual is None
            or embedding_version
            == versao_atual
        ):
            saida.append(item)

    return saida

## 11. Deduplicação

Com multi-query, é normal recuperar o mesmo trecho mais de uma vez.

Aplicamos:
1. deduplicação textual;
2. deduplicação aproximada por Jaccard.

In [13]:
# Objetivo: evitar que textos repetidos ocupem o espaço limitado do contexto.
# Primeiro detectamos igualdade do texto normalizado com um hash; depois medimos
# semelhança por conjuntos de palavras com Jaccard.
# Um set é um conjunto sem repetições: serve para lembrar hashes e comparar termos.
# Uma list preserva ordem: serve para devolver os candidatos na ordem recebida.

# Obtém palavras informativas para comparar perguntas e trechos.
# Removemos palavras frequentes e termos curtos para reduzir coincidências pouco úteis.
# Um set faz cada palavra contar uma vez, mesmo que ela apareça várias vezes no texto.
def termos(
    texto: str,
) -> set[str]:
    stopwords = {
        "de", "da", "do", "das", "dos",
        "a", "o", "as", "os", "e", "em",
        "para", "por", "com", "que", "qual",
        "quais", "um", "uma", "no", "na",
    }

    return {
        termo
        for termo in normalizar(
            texto
        ).split()
        if (
            len(termo) > 2
            and termo not in stopwords
        )
    }


# Remove trechos repetidos ou muito parecidos antes de enviá-los ao LLM.
# Mantém um representante de textos iguais ou com alta sobreposição de termos.
# O hash detecta igualdade rapidamente; Jaccard trata casos parecidos.
# O primeiro item aceito é preservado, portanto a ordem de entrada influencia a seleção.
# MD5 é usado só como resumo do texto para deduplicação, não como proteção criptográfica.
def deduplicar(
    candidatos: list[ResultadoVetorial],
    limiar_jaccard: float = 0.84,
) -> list[ResultadoVetorial]:
    saida = []
    assinaturas = []
    hashes = set()

    for item in candidatos:
        texto_norm = normalizar(
            item.texto
        )

        # O hash detecta cópias exatamente iguais de forma rápida.
        h = hashlib.md5(
            texto_norm.encode(
                "utf-8"
            )
        ).hexdigest()

        if h in hashes:
            continue

        assinatura = termos(
            item.texto
        )

        duplicado = False

        # Jaccard mede quanto os conjuntos de termos se sobrepõem.
        for outra in assinaturas:
            # Para conjuntos, | calcula união e & calcula interseção.
            # Jaccard divide quantos termos são comuns por quantos existem no total.
            # "or 1" evita divisão por zero quando ambos os conjuntos ficam vazios.
            uniao = len(
                assinatura | outra
            ) or 1

            jaccard = (
                len(
                    assinatura & outra
                )
                / uniao
            )

            if (
                jaccard
                >= limiar_jaccard
            ):
                duplicado = True
                # break encerra apenas este laço de comparação: já encontramos uma duplicata.
                # O laço externo continua verificando os próximos candidatos.
                break

        if not duplicado:
            hashes.add(h)
            assinaturas.append(
                assinatura
            )
            saida.append(item)

    return saida

## 12. Reranking

Para não introduzir outro modelo, o reranking combina:
- similaridade vetorial;
- score de RRF;
- cobertura lexical da pergunta.

Em produção poderíamos substituir isso por um cross-encoder ou reranker dedicado.

In [14]:
# Objetivo: colocar os trechos mais promissores no início da lista.
# O score combina proximidade vetorial, posição nas buscas e termos da pergunta.
# Essa implementação é uma regra numérica transparente, não um segundo modelo neural.
# Os pesos 5.0 e 0.20 são escolhas didáticas; alterá-los muda a importância relativa
# dos sinais e exigiria comparar resultados para avaliar se houve melhoria.

# Reordena os candidatos combinando similaridade vetorial, RRF e cobertura
# dos termos importantes da pergunta.
# Calcula um score composto e devolve candidatos do maior para o menor.
# Definir score dentro de rerank permite acessar termos_pergunta sem outro parâmetro.
# sorted cria uma nova lista, preservando a lista original recebida.
def rerank(
    pergunta: str,
    candidatos: list[ResultadoVetorial],
) -> list[ResultadoVetorial]:
    termos_pergunta = termos(
        pergunta
    )

    # Calcula a pontuação de um único candidato para a ordenação.
    # A cobertura conta quais termos da pergunta aparecem no trecho.
    # A soma ponderada é uma heurística da aula, não uma probabilidade de resposta correta.
    def score(
        item: ResultadoVetorial,
    ) -> float:
        termos_texto = termos(
            item.texto
        )

        # Cobertura é a fração dos termos da pergunta presentes no trecho.
        cobertura = (
            len(
                termos_pergunta
                & termos_texto
            )
            / (
                len(
                    termos_pergunta
                )
                or 1
            )
        )

        return (
            item.similaridade
            + 5.0 * item.rrf_score
            + 0.20 * cobertura
        )

    # key=score passa a função, sem chamá-la aqui. sorted a chama para cada item.
    # score() com parênteses tentaria executar a função imediatamente, sem candidato.
    return sorted(
        candidatos,
        key=score,
        reverse=True,
    )

## 13. Compressão

Selecionamos apenas sentenças relacionadas à pergunta.

A compressão aqui é extrativa para manter a aula rápida.

O LLM será usado depois para avaliar suficiência e gerar a resposta final.

In [15]:
# Objetivo: converter candidatos de busca em evidências curtas e rastreáveis.
# Aplicamos filtros antes de ordenar e comprimir para não gastar contexto com itens
# descartáveis. Depois escolhemos até top_n evidências.
# A compressão seleciona frases existentes no texto, sem pedir ao LLM para reescrevê-las.
# Guardar parte do texto original nos metadados ajuda a revisar o que foi omitido.

# Seleciona frases que compartilham termos com a pergunta.
# A interseção de conjuntos identifica palavras comuns sem pedir um resumo ao modelo.
# Quando nada coincide, usamos a primeira frase para não produzir texto vazio.
# Esse recorte pode perder condições importantes; compare-o com o trecho original.
def comprimir_trecho(
    pergunta: str,
    texto: str,
    limite_sentencas: int = 3,
) -> str:
    termos_pergunta = termos(
        pergunta
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto,
    ).strip()

    # A expressão procura espaço após ponto, exclamação ou interrogação.
    # É uma separação simples de frases; abreviações e pontuação de PDFs podem confundi-la.
    sentencas = re.split(
        r"(?<=[.!?])\s+",
        texto,
    )

    selecionadas = []

    for sentenca in sentencas:
        termos_sentenca = termos(
            sentenca
        )

        if (
            termos_pergunta
            & termos_sentenca
        ):
            selecionadas.append(
                sentenca
            )

    if not selecionadas:
        selecionadas = (
            sentencas[:1]
        )

    return " ".join(
        selecionadas[
            :limite_sentencas
        ]
    )


# Aplica filtros, remove repetições, reordena e prepara evidências.
# Filtrar primeiro evita que itens inadequados ocupem as vagas de top_n.
# Os metadados preservam scores e origem para explicar por que um trecho foi selecionado.
def pos_processar_vetor(
    pergunta: str,
    candidatos: list[ResultadoVetorial],
    top_n: int = 5,
) -> list[Evidencia]:
    # Reatribuímos candidatos após cada transformação para encadear as etapas.
    # Cada chamada recebe o resultado da anterior; mudar a ordem pode mudar a seleção.
    candidatos = filtrar_score(
        candidatos
    )

    candidatos = filtrar_versao(
        candidatos
    )

    candidatos = deduplicar(
        candidatos
    )

    candidatos = rerank(
        pergunta,
        candidatos,
    )

    evidencias = []

    for item in candidatos[
        :top_n
    ]:
        evidencias.append(
            Evidencia(
                tipo="vetor",
                conteudo=comprimir_trecho(
                    pergunta,
                    item.texto,
                ),
                fonte=item.fonte,
                metadata={
                    # Dentro de um dicionário, ** copia as chaves/valores do dicionário existente.
                    # As chaves declaradas depois acrescentam ou substituem valores nessa nova estrutura.
                    **item.metadata,
                    "similaridade": round(
                        item.similaridade,
                        4,
                    ),
                    "rrf_score": round(
                        item.rrf_score,
                        6,
                    ),
                    "consulta_origem": (
                        item.consulta_origem
                    ),
                    # Mantemos também o chunk original para
                    # auditoria e avaliação de suficiência.
                    "texto_original": item.texto[:1800],
                },
            )
        )

    return evidencias

### Compare busca simples vs pipeline vetorial avançado

In [16]:
# Objetivo: comparar uma busca com a pergunta original e uma busca com variações.
# Usamos a mesma pergunta e o mesmo escopo para facilitar a comparação.
# "avancados, pre = ..." desempacota dois valores retornados pela função.
# As impressões mostram scores e fontes; elas ajudam na inspeção, mas não demonstram
# sozinhas que a busca avançada melhorou a resposta. Leia também os trechos recuperados.

pergunta = (
    "O que Bancos de Dados aborda "
    "sobre processamento de consultas?"
)

simples = buscar_uma_consulta(
    pergunta,
    escopo="disciplina",
    k=5,
)

avancados, pre = recuperar_multi_query(
    pergunta,
    escopo="disciplina",
)

evidencias = pos_processar_vetor(
    pergunta,
    avancados,
)

print("BUSCA SIMPLES:")
for item in simples[:3]:
    print(
        round(
            item.similaridade,
            4,
        ),
        item.fonte,
    )

print("\nPIPELINE AVANÇADO:")
for e in evidencias:
    print(
        e.metadata["similaridade"],
        e.metadata["rrf_score"],
        e.fonte,
    )

BUSCA SIMPLES:
0.5204 ci1218.pdf p.1
0.5183 ci1218.pdf p.2
0.4962 ci1218.pdf p.1

PIPELINE AVANÇADO:
0.5943 0.098361 ci1218.pdf p.1
0.5579 0.094998 ci1218.pdf p.1
0.5748 0.096518 ci1218.pdf p.2
0.506 0.078621 ci1218.pdf p.2
0.4993 0.061538 ci1218.pdf p.2


# PARTE V — Roteamento

### Roteador para modelos pequenos

Um modelo de 1.5B pode falhar quando pedimos JSON livre + justificativa. Nesta versão, casos óbvios usam guardrails determinísticos e o LLM é usado apenas quando necessário, escolhendo uma **label fechada**. Isso mantém o LLM no pipeline sem permitir que ele invente uma rota fora do contrato.

## 14. Qual mecanismo deve responder?

O roteador não responde à pergunta.

Ele classifica a operação:

- `vetor`;
- `sql`;
- `grafo`;
- `vetor_sql`;
- `vetor_grafo`.

In [17]:
# Objetivo: escolher qual mecanismo deve recuperar os dados para a pergunta.
# Regras tratam operações claras; o LLM classifica os demais casos em cinco rotas.
# Retornamos um plano com rota, escopo e motivo para explicar a decisão posteriormente.
# A ordem dos if importa: pedidos híbridos precisam ser reconhecidos antes de uma
# regra mais geral para valores exatos, ou a parte semântica da pergunta seria ignorada.

ROTAS = {
    "vetor",
    "sql",
    "grafo",
    "vetor_sql",
    "vetor_grafo",
}


# Procura pistas do tema na pergunta e devolve escopo ou None.
# None significa 'não identificado' e não 'acesso liberado'.
# A função orienta a recuperação, enquanto as funções de permissão verificam o perfil.
def inferir_escopo_regra(
    pergunta: str,
) -> str | None:
    p = normalizar(pergunta)

    if any(
        termo in p
        for termo in [
            "estagio",
            "regulamento",
            "pet",
        ]
    ):
        return "estagio"

    if any(
        termo in p
        for termo in [
            "ci1218",
            "bancos de dados",
            "disciplina",
        ]
    ):
        return "disciplina"

    if any(
        termo in p
        for termo in [
            "tcc",
            "tcc1",
            "tcc2",
            "curso",
            "ppc",
            "atividades formativas",
        ]
    ):
        return "curso"

    return None


# Reconhece pedidos de relações, valores exatos e conteúdo com palavras-chave.
# Os casos híbridos precedem regras gerais para não perder a necessidade de descobrir uma entidade.
# None indica que nenhuma regra decidiu e permite que o chamador consulte o LLM.
def rotear_por_regras(
    pergunta: str,
) -> dict | None:
    """
    Guardrails determinísticos para casos óbvios.


    Evitamos que um modelo pequeno escolha uma rota impossível
    para perguntas que já revelam claramente a operação.
    """

    p = normalizar(pergunta)
    escopo = inferir_escopo_regra(pergunta)

    termos_relacao = [
        "pre requisito",
        "pre requisitos",
        "prerequisito",
        "prerequisitos",
        "depende de",
        "dependencia",
        "referencia",
        "relacao",
        "relacionamento",
        "caminho",
        "pode validar como",
        "pode ser validado como",
        "substitui",
        "substituir",
    ]

    termos_exatos = [
        "carga horaria",
        "quantos",
        "quantas",
        "total",
        "soma",
        "media",
        "maior",
        "menor",
        "ano",
        "versao",
        "frequencia",
        "percentual",
    ]

    termos_semanticos = [
        "aborda",
        "trata",
        "fala",
        "sobre",
        "conteudo",
        "ementa",
        "explica",
        "descreve",
    ]

    descritor_sem_entidade = [
        "disciplina que",
        "documento que",
        "regra que",
        "componente que",
    ]

    tem_relacao = any(
        termo in p
        for termo in termos_relacao
    )

    tem_exato = any(
        termo in p
        for termo in termos_exatos
    )

    tem_semantico = any(
        termo in p
        for termo in termos_semanticos
    )

    precisa_descobrir_entidade = any(
        termo in p
        for termo in descritor_sem_entidade
    )

    # Relação explícita conhecida -> grafo.
    # and exige ambas as condições; not nega a segunda.
    # Aqui só vamos direto ao grafo se a relação estiver clara e não houver
    # necessidade detectada de descobrir a entidade pela descrição.
    if tem_relacao and not precisa_descobrir_entidade:
        return {
            "rota": "grafo",
            "escopo": escopo,
            "motivo": (
                "a pergunta pede uma relação explícita "
                "entre entidades"
            ),
            "origem_decisao": "regra",
        }

    # Relação + entidade ainda descrita semanticamente.
    if tem_relacao and precisa_descobrir_entidade:
        return {
            "rota": "vetor_grafo",
            "escopo": escopo,
            "motivo": (
                "é necessário identificar semanticamente "
                "a entidade e depois percorrer uma relação"
            ),
            "origem_decisao": "regra",
        }

    # Valor exato/agregação + entidade semanticamente descrita.
    if tem_exato and precisa_descobrir_entidade:
        return {
            "rota": "vetor_sql",
            "escopo": escopo,
            "motivo": (
                "é necessário identificar semanticamente "
                "a entidade e depois consultar um valor estruturado"
            ),
            "origem_decisao": "regra",
        }

    # Valor exato/agregação com entidade já explícita.
    if tem_exato:
        return {
            "rota": "sql",
            "escopo": escopo,
            "motivo": (
                "a pergunta pede valor exato, agregação "
                "ou comparação estruturada"
            ),
            "origem_decisao": "regra",
        }

    # Conteúdo/explicação sem cálculo e sem relação.
    if tem_semantico:
        return {
            "rota": "vetor",
            "escopo": escopo,
            "motivo": (
                "a pergunta é predominantemente semântica"
            ),
            "origem_decisao": "regra",
        }

    return None


# Converte variantes como 'vector sql' no nome de rota usado pelo programa.
# As variantes híbridas vêm antes de 'sql' e 'vetor', pois contêm essas palavras.
# Sem essa ordem uma resposta híbrida poderia ser classificada como simples.
def normalizar_rota_llm(
    texto: str,
) -> str | None:
    """
    Aceita pequenas variações do modelo e converte
    para uma das cinco rotas válidas.
    """

    bruto = normalizar(
        limpar_bloco(texto)
    )

    mapa = [
        ("vetor_sql", "vetor_sql"),
        ("vetor sql", "vetor_sql"),
        ("vector sql", "vetor_sql"),
        ("vetor_grafo", "vetor_grafo"),
        ("vetor grafo", "vetor_grafo"),
        ("vector graph", "vetor_grafo"),
        ("grafo", "grafo"),
        ("graph", "grafo"),
        ("sql", "sql"),
        ("vetor", "vetor"),
        ("vector", "vetor"),
    ]

    for padrao, rota in mapa:
        if padrao in bruto:
            return rota

    return None


# Obtém um plano por regras ou, se necessário, por classificação do LLM.
# O conjunto fechado de rotas evita que um nome inventado selecione código inexistente.
# Se a classificação não for reconhecida, escolhe vetor; isso mantém o fluxo,
# mas não garante que essa seja a rota adequada para a pergunta.
def rotear_llm(
    pergunta: str,
) -> dict:
    """
    Roteador robusto:
    1. resolve casos óbvios por guardrails;
    2. usa o LLM somente nos casos realmente ambíguos;
    3. força o LLM a escolher UMA label fechada;
    4. valida a saída antes de usá-la.
    """

    plano_regra = rotear_por_regras(
        pergunta
    )

    # is not None distingue um plano retornado da ausência de decisão.
    # return encerra a função, então os casos resolvidos não chegam à chamada do LLM.
    if plano_regra is not None:
        return plano_regra

    system = """
Classifique a pergunta em EXATAMENTE UMA das labels abaixo.

LABELS:
vetor
sql
grafo
vetor_sql
vetor_grafo

DEFINIÇÕES:
vetor = conteúdo, significado, explicação ou trecho textual.
sql = número exato, soma, contagem, média, carga horária,
      percentual, ano, versão, máximo ou mínimo.
grafo = pré-requisito, dependência, referência, caminho
        ou relação explícita entre entidades.
vetor_sql = primeiro descobrir semanticamente uma entidade;
            depois consultar/calcular um valor estruturado.
vetor_grafo = primeiro descobrir semanticamente uma entidade;
              depois seguir uma relação explícita.

IMPORTANTE:
- não responda à pergunta;
- não explique;
- não produza JSON;
- retorne SOMENTE UMA label da lista.
""".strip()

    texto = chamar_llm(
        system,
        pergunta,
        max_new_tokens=20,
    )

    rota = normalizar_rota_llm(
        texto
    )

    if rota not in ROTAS:
        # Fallback seguro para conteúdo textual.
        rota = "vetor"

    return {
        "rota": rota,
        "escopo": inferir_escopo_regra(
            pergunta
        ),
        "motivo": (
            "classificação por LLM em conjunto fechado de rotas"
        ),
        "origem_decisao": "llm",
        "saida_bruta_llm": texto,
    }

### Teste do roteador antes de executar as rotas

In [18]:
# Objetivo: observar decisões do roteador sem executar as consultas escolhidas.
# Uma lista reúne perguntas de naturezas diferentes para comparar os planos.
# O for repete a mesma chamada para cada pergunta e imprime rota, escopo e motivo.
# Observe origem_decisao: o nome rotear_llm não implica que houve uso de IA;
# casos reconhecidos pelas regras retornam antes de chamar o modelo.

perguntas = [
    (
        "O que a disciplina de Bancos "
        "de Dados aborda sobre transações?"
    ),
    (
        "Qual é a carga horária total "
        "dos TCCs?"
    ),
    (
        "Quais são os pré-requisitos "
        "da CI1218?"
    ),
    (
        "Qual é a carga horária da disciplina "
        "que trata de processamento de consultas?"
    ),
]

for pergunta in perguntas:
    print("\n", pergunta)

    print(
        json.dumps(
            rotear_llm(
                pergunta
            ),
            ensure_ascii=False,
            indent=2,
        )
    )


 O que a disciplina de Bancos de Dados aborda sobre transações?
{
  "rota": "vetor",
  "escopo": "disciplina",
  "motivo": "a pergunta é predominantemente semântica",
  "origem_decisao": "regra"
}

 Qual é a carga horária total dos TCCs?
{
  "rota": "sql",
  "escopo": "curso",
  "motivo": "a pergunta pede valor exato, agregação ou comparação estruturada",
  "origem_decisao": "regra"
}

 Quais são os pré-requisitos da CI1218?
{
  "rota": "grafo",
  "escopo": "disciplina",
  "motivo": "a pergunta pede uma relação explícita entre entidades",
  "origem_decisao": "regra"
}

 Qual é a carga horária da disciplina que trata de processamento de consultas?
{
  "rota": "vetor_sql",
  "escopo": "disciplina",
  "motivo": "é necessário identificar semanticamente a entidade e depois consultar um valor estruturado",
  "origem_decisao": "regra"
}


### Teste de regressão do roteador

As quatro perguntas abaixo devem resultar, respectivamente, em `vetor`, `sql`, `grafo` e `vetor_sql`.

In [19]:
# Objetivo: detectar mudanças inesperadas no roteamento de perguntas conhecidas.
# Cada tupla guarda um par (pergunta, rota esperada). O for desempacota esse par.
# O operador == compara valores e produz True ou False; = faz uma atribuição.
# O OK exibido verifica apenas a rota: nenhuma resposta final é avaliada aqui.
# Chamamos isso de teste de regressão porque repetir os casos ajuda a perceber
# se uma alteração futura mudou um comportamento que desejávamos preservar.

casos_esperados = [
    (
        "O que a disciplina de Bancos de Dados aborda sobre transações?",
        "vetor",
    ),
    (
        "Qual é a carga horária total do TCC1 e do TCC2?",
        "sql",
    ),
    (
        "Quais são os pré-requisitos da CI1218?",
        "grafo",
    ),
    (
        "Qual é a carga horária da disciplina que trata de processamento de consultas?",
        "vetor_sql",
    ),
]

for pergunta, esperado in casos_esperados:
    plano = rotear_llm(pergunta)
    obtido = plano["rota"]

    print("\nPERGUNTA:", pergunta)
    print("esperado:", esperado)
    print("obtido:  ", obtido)
    print("OK:", obtido == esperado)
    print("origem:", plano.get("origem_decisao"))


PERGUNTA: O que a disciplina de Bancos de Dados aborda sobre transações?
esperado: vetor
obtido:   vetor
OK: True
origem: regra

PERGUNTA: Qual é a carga horária total do TCC1 e do TCC2?
esperado: sql
obtido:   sql
OK: True
origem: regra

PERGUNTA: Quais são os pré-requisitos da CI1218?
esperado: grafo
obtido:   grafo
OK: True
origem: regra

PERGUNTA: Qual é a carga horária da disciplina que trata de processamento de consultas?
esperado: vetor_sql
obtido:   vetor_sql
OK: True
origem: regra


# PARTE VI — NL2SQL

## 15. Estrutura relacional derivada do corpus

A base é pequena de propósito.

Ela serve para mostrar que uma resposta estruturada pode ser mais adequada que busca vetorial para `SUM`, `COUNT` e valores exatos.

In [20]:
# Objetivo: criar uma pequena base relacional para cálculos e consultas exatas.
# Os valores são cadastrados manualmente a partir do corpus; esta célula não os
# extrai automaticamente dos PDFs. Alterar os PDFs não atualiza esses números sozinho.
# DROP TABLE recria as três tabelas didáticas a cada execução, apagando seu conteúdo anterior.
# SQL calcula somas e filtra IDs com precisão; fonte e versão registram de onde veio o dado.
# Cada tupla da lista de inserção segue a mesma ordem das colunas da tabela.

CAMINHO_SQLITE = Path(
    PASTA_RAIZ / "academico.db"
)

conn = sqlite3.connect(
    CAMINHO_SQLITE
)

# executescript executa várias instruções SQL de preparação de uma vez.
# PRIMARY KEY torna o id identificador único; INTEGER e REAL armazenam números.
# Essa preparação pode escrever/apagar dados; as consultas dos alunos são abertas
# posteriormente em conexões somente leitura.
conn.executescript("""
DROP TABLE IF EXISTS componentes;
DROP TABLE IF EXISTS documentos;
DROP TABLE IF EXISTS regras_estagio;

CREATE TABLE componentes (
    id TEXT PRIMARY KEY,
    nome TEXT,
    tipo TEXT,
    carga_horaria INTEGER,
    obrigatorio INTEGER,
    fonte TEXT,
    versao TEXT
);

CREATE TABLE documentos (
    id TEXT PRIMARY KEY,
    titulo TEXT,
    escopo TEXT,
    ano INTEGER,
    versao TEXT
);

CREATE TABLE regras_estagio (
    id TEXT PRIMARY KEY,
    regra TEXT,
    valor REAL,
    unidade TEXT,
    fonte TEXT,
    versao TEXT
);
""")

# executemany repete o INSERT para cada tupla da lista.
# Os sete ? recebem os sete valores na ordem das colunas, sem montar SQL
# concatenando conteúdo. Isso separa a estrutura da consulta dos dados inseridos.
conn.executemany(
    """
    INSERT INTO componentes
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """,
    [
        (
            "CURSO",
            "Bacharelado em Ciência da Computação",
            "curso",
            3200,
            1,
            "PPC",
            "2018",
        ),
        (
            "CI1218",
            "Bancos de Dados",
            "disciplina",
            60,
            1,
            "Ficha CI1218",
            "ficha_ci1218",
        ),
        (
            "ESTAGIO",
            "Estágio Obrigatório",
            "estagio",
            220,
            1,
            "Regulamento de Estágio",
            "2024",
        ),
        (
            "ATIVIDADES_FORMATIVAS",
            "Atividades Formativas",
            "atividade",
            160,
            1,
            "PPC",
            "2018",
        ),
        (
            "TCC1",
            "Trabalho de Conclusão de Curso 1",
            "tcc",
            150,
            1,
            "PPC",
            "2018",
        ),
        (
            "TCC2",
            "Trabalho de Conclusão de Curso 2",
            "tcc",
            150,
            1,
            "PPC",
            "2018",
        ),
    ],
)

conn.executemany(
    """
    INSERT INTO documentos
    VALUES (?, ?, ?, ?, ?)
    """,
    [
        (
            "PPC",
            "Projeto Pedagógico do Curso",
            "curso",
            2018,
            "2018",
        ),
        (
            "REG_ESTAGIO",
            "Regulamento de Estágio",
            "estagio",
            2024,
            "2024",
        ),
        (
            "FICHA_CI1218",
            "Ficha CI1218",
            "disciplina",
            None,
            "ficha_ci1218",
        ),
    ],
)

conn.executemany(
    """
    INSERT INTO regras_estagio
    VALUES (?, ?, ?, ?, ?, ?)
    """,
    [
        (
            "EST_CARGA",
            "Carga horária do estágio obrigatório",
            220,
            "horas",
            "Regulamento de Estágio, Art. 14",
            "2024",
        ),
        (
            "PET_CARGA",
            "Carga presencial de PET para validação como estágio",
            220,
            "horas",
            "Regulamento de Estágio, Art. 29",
            "2024",
        ),
    ],
)

# commit confirma as inserções da conexão para persistirem no arquivo.
# close libera a conexão; o arquivo academico.db continua disponível no disco.
conn.commit()
conn.close()

print(CAMINHO_SQLITE)

/home/ubuntu/mba-genai/CONT/aula05/academico.db


## 15.1. Teste da camada SQL antes do NL2SQL

Antes de envolver o LLM, vamos verificar se o banco estruturado está funcionando sozinho.

Esta etapa responde a uma pergunta importante de depuração:

> **O problema está no banco ou na tradução NL2SQL?**

Os testes abaixo usam SQL escrito manualmente e executam diretamente no SQLite. Se eles falharem, não adianta investigar o LLM ainda.

In [21]:
# Objetivo: conferir o banco separadamente da geração de SQL pelo modelo.
# Consultas escritas previamente verificam tabelas, registros e soma das cargas.
# Assim, um erro de dados pode ser investigado antes de avaliar o NL2SQL.
# O teste exige pelo menos uma linha por consulta; isso não valida todos os valores.
# Confira o resultado da soma manualmente: os dados cadastrados têm dois TCCs de 150h.

# Este teste separa o banco de dados do LLM: assim sabemos se os dados e
# as consultas SQL básicas funcionam antes de gerar SQL automaticamente.
# Executa consultas conhecidas e imprime dados para inspeção.
# Uma consulta que devolve linhas passa nesta checagem; números ainda precisam ser conferidos.
# finally fecha a conexão mesmo se alguma consulta lançar erro.
def testar_banco_sql_simples():
    print("=" * 70)
    print("TESTE DIRETO DO SQLITE — SEM LLM")
    print("=" * 70)

    if not CAMINHO_SQLITE.exists():
        raise FileNotFoundError(
            f"Banco não encontrado em: {CAMINHO_SQLITE}"
        )

    # mode=ro abre o banco somente para leitura, protegendo os dados de aula.
    conn = sqlite3.connect(
        f"file:{CAMINHO_SQLITE.resolve()}?mode=ro",
        uri=True,
    )

    testes = [
        (
            "1. Tabelas existentes",
            """
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            ORDER BY name;
            """,
        ),
        (
            "2. Quantidade de componentes",
            """
            SELECT COUNT(*) AS quantidade
            FROM componentes;
            """,
        ),
        (
            "3. Buscar CI1218",
            """
            SELECT id, nome, carga_horaria
            FROM componentes
            WHERE id = 'CI1218';
            """,
        ),
        (
            "4. Buscar TCC1 e TCC2",
            """
            SELECT id, nome, carga_horaria
            FROM componentes
            WHERE id IN ('TCC1', 'TCC2')
            ORDER BY id;
            """,
        ),
        (
            "5. Somar carga horária dos TCCs",
            """
            SELECT SUM(carga_horaria) AS carga_total
            FROM componentes
            WHERE id IN ('TCC1', 'TCC2');
            """,
        ),
        (
            "6. Buscar estágio obrigatório",
            """
            SELECT id, nome, carga_horaria
            FROM componentes
            WHERE id = 'ESTAGIO';
            """,
        ),
    ]

    try:
        # Cada consulta verifica um aspecto pequeno e fácil de conferir.
        for titulo, sql in testes:
            print(f"\n{titulo}")
            print("-" * 70)

            # execute devolve um cursor, que permite ler os resultados.
            # description informa as colunas; fetchall lê todas as linhas deste pequeno teste.
            cursor = conn.execute(sql)
            colunas = [item[0] for item in cursor.description]
            linhas = cursor.fetchall()

            print("SQL:")
            print(sql.strip())

            print("\nColunas:")
            print(colunas)

            print("\nResultado:")
            print(linhas)

            if not linhas:
                raise RuntimeError(
                    f"O teste '{titulo}' não retornou nenhuma linha."
                )

    finally:
        conn.close()

    print("\n" + "=" * 70)
    print("BANCO SQL OK")
    print("Todos os testes simples retornaram resultados.")
    print("=" * 70)


testar_banco_sql_simples()

TESTE DIRETO DO SQLITE — SEM LLM

1. Tabelas existentes
----------------------------------------------------------------------
SQL:
SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            ORDER BY name;

Colunas:
['name']

Resultado:
[('componentes',), ('documentos',), ('regras_estagio',)]

2. Quantidade de componentes
----------------------------------------------------------------------
SQL:
SELECT COUNT(*) AS quantidade
            FROM componentes;

Colunas:
['quantidade']

Resultado:
[(6,)]

3. Buscar CI1218
----------------------------------------------------------------------
SQL:
SELECT id, nome, carga_horaria
            FROM componentes
            WHERE id = 'CI1218';

Colunas:
['id', 'nome', 'carga_horaria']

Resultado:
[('CI1218', 'Bancos de Dados', 60)]

4. Buscar TCC1 e TCC2
----------------------------------------------------------------------
SQL:
SELECT id, nome, carga_horaria
            FROM componentes
            WHERE id IN ('TCC1'

## Decisões de engenharia


1. **Template determinístico ≠ NL2SQL por LLM.**  
   Se a entidade já foi resolvida e a operação é previsível, o sistema pode usar um template. A saída registra `origem="template_deterministico"`. Quando o modelo gera SQL, registra `origem="llm"`.

2. **IDs resolvidos entram de fato no contexto do LLM.**  


3. **Validação estrutural por AST.**  
   `sqlglot` verifica que existe uma única consulta de leitura. A conexão SQLite continua em `mode=ro` como defesa adicional.

4. **Permissão não depende apenas do roteador.**  
   A busca vetorial aplica ACL nos metadados mesmo com `escopo=None`; SQL e grafo também fazem checagem na camada de execução. Não um IAM/ABAC completo de produção.

5. **Templates usam parâmetros SQLite.**  
   IDs entram como `?` + `params`, não por interpolação manual na string SQL.


> O LLM interpreta quando há incerteza; o software determinístico controla invariantes, permissões e execução.

## 16. Linguagem natural -> SQL

> **O contexto semântico é dinâmico:** o código consulta o próprio SQLite para descobrir schema, valores categóricos e entidades candidatas antes de chamar o LLM.

In [22]:
# Objetivo: traduzir perguntas para consultas usando a estrutura real do SQLite.
# Mostrar ao modelo os IDs e categorias existentes reduz a chance de inventar nomes.
# ConsultaSQL reúne texto SQL, parâmetros e origem em um único objeto.
# Para entidades já resolvidas, uma consulta parametrizada pronta cobre operações
# simples; para outros casos o LLM gera a consulta. Registramos qual estratégia foi usada.
# Definir essas funções não executa suas consultas: a execução ocorre nas chamadas posteriores.

# Este texto descreve o esquema para leitura humana/compatibilidade.
# As funções atuais de contexto chamam obter_schema_sql para descobrir o banco real,
# em vez de depender exclusivamente desta descrição fixa.
SCHEMA_SQL = """
componentes(
  id TEXT,
  nome TEXT,
  tipo TEXT,
  carga_horaria INTEGER,
  obrigatorio INTEGER,
  fonte TEXT,
  versao TEXT
)

documentos(
  id TEXT,
  titulo TEXT,
  escopo TEXT,
  ano INTEGER,
  versao TEXT
)

regras_estagio(
  id TEXT,
  regra TEXT,
  valor REAL,
  unidade TEXT,
  fonte TEXT,
  versao TEXT
)
""".strip()


@dataclass
class ConsultaSQL:
    """
    Representa SQL + parâmetros + proveniência.

    origem:
      - "llm"
      - "template_deterministico"
      - "llm_reparo"
    """
    sql: str
    params: tuple[Any, ...] = ()
    origem: str = "llm"

    # Define o texto mostrado quando usamos print em um objeto ConsultaSQL.
    # Devolver só self.sql facilita ler a consulta; params e origem continuam disponíveis
    # como atributos separados do objeto, por exemplo consulta.params.
    def __str__(self) -> str:
        return self.sql


TIPO_PARA_ESCOPO = {
    "curso": "curso",
    "disciplina": "disciplina",
    "estagio": "estagio",
    "atividade": "curso",
    "tcc": "curso",
}


# Lê nomes de tabelas e colunas diretamente do SQLite.
# PRAGMA table_info fornece metadados estruturais para orientar a geração.
# Descobrir o esquema reduz divergências entre uma descrição escrita à mão e o banco real.
def obter_schema_sql() -> str:
    """
    Descobre a estrutura real do SQLite em tempo de execução.
    """
    conn = sqlite3.connect(
        f"file:{CAMINHO_SQLITE.resolve()}?mode=ro",
        uri=True,
    )

    try:
        tabelas = [
            linha[0]
            for linha in conn.execute(
                """
                SELECT name
                FROM sqlite_master
                WHERE type = 'table'
                ORDER BY name
                """
            ).fetchall()
        ]

        blocos = []

        for tabela in tabelas:
            colunas = conn.execute(
                f"PRAGMA table_info({tabela})"
            ).fetchall()

            descricao = ", ".join(
                f"{coluna[1]} {coluna[2]}"
                for coluna in colunas
            )

            blocos.append(
                f"{tabela}({descricao})"
            )

        return "\n".join(blocos)

    finally:
        conn.close()


# Extrai termos que podem identificar registros, como TCC1.
# Remove palavras sobre a operação ('total', 'carga') para priorizar a entidade.
# A remoção de repetidos preserva a ordem para manter a inspeção previsível.
def termos_para_metadata(
    pergunta: str,
) -> list[str]:
    tokens = re.findall(
        r"[A-Za-zÀ-ÿ0-9_]+",
        pergunta,
    )

    stopwords = {
        "qual", "quais", "quanto", "quantos",
        "a", "o", "as", "os",
        "de", "da", "do", "das", "dos",
        "e", "em", "no", "na",
        "é", "são", "total",
        "carga", "horária", "horaria",
    }

    termos = []

    for token in tokens:
        token = token.strip()

        if (
            len(token) >= 3
            and token.lower() not in stopwords
        ):
            termos.append(token)

    return list(
        dict.fromkeys(termos)
    )


# Busca categorias e registros relacionados aos termos da pergunta.
# Apesar do nome, esta etapa usa LIKE por correspondência textual, sem embeddings.
# O limite evita mandar registros demais ao modelo. Os ? recebem valores separados,
# de modo que o termo buscado é tratado como dado da consulta.
def obter_contexto_semantico_sql(
    pergunta: str,
    limite: int = 12,
) -> str:
    """
    Recupera valores relevantes diretamente do banco.
    """
    conn = sqlite3.connect(
        f"file:{CAMINHO_SQLITE.resolve()}?mode=ro",
        uri=True,
    )

    try:
        tipos = [
            linha[0]
            for linha in conn.execute(
                """
                SELECT DISTINCT tipo
                FROM componentes
                ORDER BY tipo
                """
            ).fetchall()
        ]

        termos = termos_para_metadata(
            pergunta
        )

        candidatos = []

        for termo in termos:
            # No LIKE do SQLite, % significa qualquer sequência de caracteres.
            # Assim '%TCC1%' encontra valores que contêm TCC1; COLLATE NOCASE ignora caixa
            # conforme as regras do SQLite, sem equivaler à nossa normalização de acentos.
            padrao = f"%{termo}%"

            linhas = conn.execute(
                """
                SELECT
                    id,
                    nome,
                    tipo,
                    carga_horaria,
                    fonte,
                    versao
                FROM componentes
                WHERE id LIKE ? COLLATE NOCASE
                   OR nome LIKE ? COLLATE NOCASE
                   OR tipo LIKE ? COLLATE NOCASE
                LIMIT ?
                """,
                (
                    padrao,
                    padrao,
                    padrao,
                    limite,
                ),
            ).fetchall()

            candidatos.extend(
                linhas
            )

        vistos = set()
        unicos = []

        for linha in candidatos:
            if linha not in vistos:
                vistos.add(linha)
                unicos.append(linha)

        unicos = unicos[:limite]

        linhas_texto = [
            (
                f"id={linha[0]!r}, "
                f"nome={linha[1]!r}, "
                f"tipo={linha[2]!r}, "
                f"carga_horaria={linha[3]!r}, "
                f"fonte={linha[4]!r}, "
                f"versao={linha[5]!r}"
            )
            for linha in unicos
        ]

        return f"""
VALORES CATEGÓRICOS OBSERVADOS:
tipo = {tipos}

ENTIDADES/VALORES RELEVANTES:
{chr(10).join(linhas_texto) if linhas_texto else "(nenhum candidato encontrado)"}
""".strip()

    finally:
        conn.close()


# Reúne esquema, registros candidatos e IDs resolvidos em um prompt de apoio.
# Fornecer IDs já descobertos evita pedir ao modelo que identifique a entidade de novo.
# Esse texto orienta a geração; as validações posteriores ainda precisam conferir o SQL.
def montar_contexto_nl2sql(
    pergunta: str,
    contexto_extra: str = "",
    ids_resolvidos: list[str] | None = None,
) -> str:
    """
    ids_resolvidos agora é efetivamente usado.
    """
    schema_real = obter_schema_sql()

    contexto_semantico = (
        obter_contexto_semantico_sql(
            pergunta
        )
    )

    bloco_ids = "(nenhum)"

    if ids_resolvidos:
        bloco_ids = (
            "IDs já resolvidos por etapas anteriores: "
            + ", ".join(ids_resolvidos)
            + ". Use-os diretamente; não redescubra a entidade pelo nome."
        )

    return f"""
SCHEMA REAL:
{schema_real}

METADADOS/VALORES RECUPERADOS:
{contexto_semantico}

IDs RESOLVIDOS:
{bloco_ids}

CONTEXTO EXTRA:
{contexto_extra or "(nenhum)"}
""".strip()


# Pede ao LLM uma consulta e a encapsula com origem='llm'.
# A função gera SQL, mas não o executa. Essa separação permite inspecionar e validar
# a proposta antes de acessar resultados do banco.
def nl2sql(
    pergunta: str,
    contexto_extra: str = "",
    ids_resolvidos: list[str] | None = None,
) -> ConsultaSQL:
    """
    NL2SQL generativo.

    Esta função SEMPRE chama o LLM.
    Se o pipeline escolher um template determinístico,
    isso acontece em nl2sql_com_ids() e fica registrado
    em ConsultaSQL.origem.
    """
    system = """
Você é um tradutor NL2SQL para SQLite.

Regras:
- retorne APENAS uma consulta SELECT;
- use somente tabelas e colunas mostradas no schema;
- use os valores reais recuperados nos metadados;
- não invente valores categóricos;
- se houver IDs já resolvidos, use-os diretamente;
- use agregações quando a pergunta exigir;
- nunca altere dados;
- sem explicações e sem Markdown.
""".strip()

    contexto = montar_contexto_nl2sql(
        pergunta,
        contexto_extra=contexto_extra,
        ids_resolvidos=ids_resolvidos,
    )

    user = f"""
{contexto}

PERGUNTA:
{pergunta}

SQL:
""".strip()

    sql = limpar_bloco(
        chamar_llm(
            system,
            user,
            max_new_tokens=180,
        )
    )

    return ConsultaSQL(
        sql=sql,
        params=(),
        origem="llm",
    )


# Confirma que os IDs existem e pertencem aos escopos autorizados.
# A consulta IN recebe um ? por ID; os valores são enviados separadamente.
# O conjunto facilita verificar autorização e a lista final mantém a ordem da entrada.
def validar_ids_no_banco(
    ids: list[str],
    perfil: str = "publico",
) -> list[str]:
    """
    Valida IDs no banco e aplica permissão no nível da linha.
    """
    if not ids:
        return []

    permitidos = escopos_permitidos(
        perfil
    )

    conn = sqlite3.connect(
        f"file:{CAMINHO_SQLITE.resolve()}?mode=ro",
        uri=True,
    )

    try:
        # Um ? por ID faz a consulta aceitar uma lista de tamanho variável.
        # A string interpolada contém apenas os marcadores; os valores reais vão à parte
        # em conn.execute. Isso evita tratar o conteúdo de um ID como instrução SQL.
        placeholders = ",".join(
            "?"
            for _ in ids
        )

        linhas = conn.execute(
            f"""
            SELECT id, tipo
            FROM componentes
            WHERE id IN ({placeholders})
            """,
            ids,
        ).fetchall()

        ids_autorizados = set()

        for identificador, tipo in linhas:
            escopo = TIPO_PARA_ESCOPO.get(
                tipo
            )

            if escopo in permitidos:
                ids_autorizados.add(
                    identificador
                )

        return [
            identificador
            for identificador in ids
            if identificador in ids_autorizados
        ]

    finally:
        conn.close()


# Restringe candidatos ao tipo solicitado e às permissões do perfil.
# Se a pergunta pede disciplina, removemos candidatos que representam o curso inteiro.
# Isso evita usar a carga horária do curso ao responder sobre uma disciplina.
def refinar_ids_por_pergunta(
    pergunta: str,
    ids: list[str],
    perfil: str = "publico",
) -> list[str]:
    """
    Refina candidatos pela intenção e pela permissão.
    """
    if not ids:
        return []

    p = normalizar(
        pergunta
    )

    tipo_esperado = None

    if "disciplina" in p:
        tipo_esperado = "disciplina"
    elif "curso" in p:
        tipo_esperado = "curso"
    elif "estagio" in p:
        tipo_esperado = "estagio"
    elif "tcc" in p:
        tipo_esperado = "tcc"

    ids_validos = validar_ids_no_banco(
        ids,
        perfil=perfil,
    )

    if (
        not tipo_esperado
        or not ids_validos
    ):
        return ids_validos

    conn = sqlite3.connect(
        f"file:{CAMINHO_SQLITE.resolve()}?mode=ro",
        uri=True,
    )

    try:
        placeholders = ",".join(
            "?"
            for _ in ids_validos
        )

        linhas = conn.execute(
            f"""
            SELECT id
            FROM componentes
            WHERE id IN ({placeholders})
              AND tipo = ?
            """,
            # *ids_validos espalha os IDs, e tipo_esperado ocupa o último parâmetro.
            # A ordem precisa corresponder aos ? de IN (...) e, depois, de tipo = ?.
            [
                *ids_validos,
                tipo_esperado,
            ],
        ).fetchall()

        encontrados = {
            linha[0]
            for linha in linhas
        }

        return [
            identificador
            for identificador in ids_validos
            if identificador in encontrados
        ]

    finally:
        conn.close()


# Monta uma consulta pronta quando IDs e operação são suficientemente conhecidos.
# Um template oferece SQL previsível sem custo de geração pelo modelo.
# SUM é escolhido para total com vários IDs; outros casos cobertos listam cargas individuais.
# Retornar None indica que o chamador deve recorrer à geração NL2SQL.
def construir_sql_template_entidades_resolvidas(
    pergunta: str,
    ids: list[str],
) -> ConsultaSQL | None:
    """
    TEMPLATE DETERMINÍSTICO, não NL2SQL por LLM.

    Uso deliberado:
    quando a entidade já foi resolvida e a operação é previsível,
    o template tem menor custo e maior previsibilidade.

    Os IDs entram como parâmetros SQLite (?), nunca por interpolação.
    """
    if not ids:
        return None

    p = normalizar(
        pergunta
    )

    placeholders = ",".join(
        "?"
        for _ in ids
    )

    if "carga horaria" in p:
        if (
            any(
                termo in p
                for termo in [
                    "total",
                    "soma",
                    "somar",
                ]
            )
            and len(ids) > 1
        ):
            return ConsultaSQL(
                sql=f"""
SELECT
    SUM(carga_horaria) AS carga_horaria_total,
    GROUP_CONCAT(id, ', ') AS componentes
FROM componentes
WHERE id IN ({placeholders});
""".strip(),
                params=tuple(ids),
                origem="template_deterministico",
            )

        return ConsultaSQL(
            sql=f"""
SELECT
    id,
    nome,
    carga_horaria,
    fonte,
    versao
FROM componentes
WHERE id IN ({placeholders});
""".strip(),
            params=tuple(ids),
            origem="template_deterministico",
        )

    return None


# Alias apenas para notebooks/células antigas.
construir_sql_com_entidades_resolvidas = (
    construir_sql_template_entidades_resolvidas
)


# Coordena o refinamento dos IDs e a escolha entre template e LLM.
# Se nenhum candidato válido restar, lança erro em vez de consultar a entidade errada.
# Devolve consulta e IDs finais para permitir verificar ambos nas etapas seguintes.
def nl2sql_com_ids(
    pergunta: str,
    ids: list[str],
    perfil: str = "publico",
) -> tuple[ConsultaSQL, list[str]]:
    """
    Rota vetor + SQL.

    A proveniência deixa explícito se houve:
    - template determinístico; ou
    - geração pelo LLM.
    """
    demo_log(
        "[DEMO vetor_sql] IDs candidatos:",
        ids,
    )

    ids_finais = refinar_ids_por_pergunta(
        pergunta,
        ids,
        perfil=perfil,
    )

    demo_log(
        "[DEMO vetor_sql] IDs finais:",
        ids_finais,
    )

    if not ids_finais:
        raise ValueError(
            "Nenhuma entidade autorizada e compatível "
            "permaneceu após o refinamento."
        )

    consulta = (
        construir_sql_template_entidades_resolvidas(
            pergunta,
            ids_finais,
        )
    )

    # O template cobreu a operação, então devolvemos o SQL sem chamar o LLM.
    # Registrar consulta.origem permite explicar no relatório qual mecanismo foi usado.
    if consulta is not None:
        demo_log(
            "[DEMO vetor_sql] estratégia:",
            consulta.origem,
            "(LLM não foi chamado para gerar o SQL)",
        )
        return consulta, ids_finais

    consulta = nl2sql(
        pergunta,
        contexto_extra=(
            "A entidade já foi resolvida pela recuperação."
        ),
        ids_resolvidos=ids_finais,
    )

    demo_log(
        "[DEMO vetor_sql] estratégia:",
        consulta.origem,
    )

    return consulta, ids_finais

## 16.1. Teste da descoberta dinâmica de metadados


> Execute primeiro a célula **16 — Linguagem natural -> SQL**, que define
`obter_schema_sql()`, `obter_contexto_semantico_sql()` e `nl2sql()`.

In [23]:
# Objetivo: mostrar quais informações do banco orientam a geração de SQL.
# Primeiro imprimimos tabelas/colunas, depois registros relacionados à pergunta,
# e só então chamamos o LLM. Confira se TCC1 e TCC2 aparecem como IDs reais.
# A consulta gerada é exibida aqui; os testes de execução vêm nas próximas células.

pergunta_metadata = (
    "Qual é a carga horária total "
    "do TCC1 e do TCC2?"
)

print("SCHEMA DESCOBERTO:")
print(obter_schema_sql())

print("\nCONTEXTO SEMÂNTICO RECUPERADO:")
print(
    obter_contexto_semantico_sql(
        pergunta_metadata
    )
)

print("\nSQL GERADO PELO LLM:")
print(
    nl2sql(
        pergunta_metadata
    )
)

SCHEMA DESCOBERTO:
componentes(id TEXT, nome TEXT, tipo TEXT, carga_horaria INTEGER, obrigatorio INTEGER, fonte TEXT, versao TEXT)
documentos(id TEXT, titulo TEXT, escopo TEXT, ano INTEGER, versao TEXT)
regras_estagio(id TEXT, regra TEXT, valor REAL, unidade TEXT, fonte TEXT, versao TEXT)

CONTEXTO SEMÂNTICO RECUPERADO:
VALORES CATEGÓRICOS OBSERVADOS:
tipo = ['atividade', 'curso', 'disciplina', 'estagio', 'tcc']

ENTIDADES/VALORES RELEVANTES:
id='TCC1', nome='Trabalho de Conclusão de Curso 1', tipo='tcc', carga_horaria=150, fonte='PPC', versao='2018'
id='TCC2', nome='Trabalho de Conclusão de Curso 2', tipo='tcc', carga_horaria=150, fonte='PPC', versao='2018'

SQL GERADO PELO LLM:
SELECT SUM(carga_horaria) AS total_carga_horaria FROM componentes WHERE nome IN ('Trabalho de Conclusão de Curso 1', 'Trabalho de Conclusão de Curso 2')


## 17. Validação, EXPLAIN e read-only

O SQL do LLM é uma **proposta**.

Antes de executar:
1. só aceitamos `SELECT`;
2. bloqueamos operações proibidas;
3. aceitamos uma única instrução;
4. executamos com conexão somente leitura;
5. coletamos `EXPLAIN QUERY PLAN`.

In [24]:
# Objetivo: verificar a consulta e transformá-la em resultados rastreáveis.
# Validamos operação pedida, presença de IDs e restrições básicas do texto SQL.
# Uma conexão somente leitura impede gravações durante a consulta.
# As checagens textuais são didáticas: não provam equivalência completa entre SQL e pergunta.
# Esta função verifica que o perfil existe, mas não acrescenta filtros de linha a SQL arbitrário;
# as checagens de escopo e de IDs presentes no restante do pipeline têm papéis separados.

# Lê os IDs existentes em vez de repetir uma lista fixa na validação.
# Cada linha SQL é uma tupla; linha[0] seleciona sua primeira coluna.
# Assim o conjunto consultado acompanha o conteúdo atual da tabela.
def obter_ids_componentes() -> list[str]:
    """
    Lê dinamicamente os IDs existentes no banco.
    Não conhece previamente CI1218, TCC1, TCC2 etc.
    """
    conn = sqlite3.connect(
        f"file:{CAMINHO_SQLITE.resolve()}?mode=ro",
        uri=True,
    )

    try:
        linhas = conn.execute(
            """
            SELECT id
            FROM componentes
            """
        ).fetchall()

        return [
            linha[0]
            for linha in linhas
        ]

    finally:
        conn.close()


# Compara a pergunta normalizada com IDs realmente cadastrados.
# Retorna uma lista, pois a pergunta pode mencionar várias entidades.
# A busca é por substring; não faz resolução semântica de nomes ou descrições.
def extrair_ids_da_pergunta(
    pergunta: str,
) -> list[str]:
    """
    Descobre se a pergunta menciona explicitamente
    algum ID que realmente existe no banco.
    """
    pergunta_norm = normalizar(pergunta)

    ids_encontrados = []

    for identificador in obter_ids_componentes():

        id_norm = normalizar(
            identificador
        )

        # Ex.: CI1218, TCC1, ESTAGIO...
        if id_norm in pergunta_norm:
            ids_encontrados.append(
                identificador
            )

    return ids_encontrados
# Relaciona termos da pergunta a sum, count, avg, max ou min.
# None representa ausência de operação reconhecida pelas regras.
# A ordem das condições define prioridade se mais de um termo aparecer na pergunta.
def inferir_operacao_sql(
    pergunta: str,
) -> str | None:

    p = normalizar(pergunta)

    if any(
        termo in p
        for termo in [
            "total",
            "soma",
            "somar",
        ]
    ):
        return "sum"

    if any(
        termo in p
        for termo in [
            "quantos",
            "quantas",
            "quantidade",
        ]
    ):
        return "count"

    if any(
        termo in p
        for termo in [
            "media",
            "média",
        ]
    ):
        return "avg"

    if any(
        termo in p
        for termo in [
            "maior",
            "maximo",
            "máximo",
        ]
    ):
        return "max"

    if any(
        termo in p
        for termo in [
            "menor",
            "minimo",
            "mínimo",
        ]
    ):
        return "min"

    return None
# Compara sinais da pergunta com operação SQL e IDs previamente resolvidos.
# Considera texto e parâmetros porque consultas com ? guardam os IDs fora do SQL.
# São verificações textuais: encontrar um ID ou SUM no texto não prova que todos
# os filtros e cálculos correspondem exatamente à intenção do usuário.
def validar_intencao_sql(
    pergunta: str,
    consulta: ConsultaSQL | str,
    ids_resolvidos: list[str] | None = None,
) -> None:
    """
    Verifica se o SQL preserva elementos importantes
    da intenção expressa na pergunta.

    Não possui conhecimento específico sobre TCC,
    CI1218, estágio ou qualquer outra entidade.
    """

    consulta = _consulta_sql(
        consulta
    )

    p = normalizar(
        pergunta
    )

    material = (
        consulta.sql.upper()
        + " "
        + " ".join(
            str(valor).upper()
            for valor in consulta.params
        )
    )

    # -----------------------------------------------------
    # 1. Operação solicitada pela pergunta
    # -----------------------------------------------------

    operacao = inferir_operacao_sql(
        pergunta
    )

    if (
        operacao == "sum"
        and "SUM(" not in material
    ):
        raise ValueError(
            "A pergunta pede uma soma/total, "
            "mas o SQL não utiliza SUM."
        )

    if (
        operacao == "count"
        and "COUNT(" not in material
    ):
        raise ValueError(
            "A pergunta pede uma quantidade, "
            "mas o SQL não utiliza COUNT."
        )

    if (
        operacao == "avg"
        and "AVG(" not in material
    ):
        raise ValueError(
            "A pergunta pede uma média, "
            "mas o SQL não utiliza AVG."
        )

    if (
        operacao == "max"
        and (
            "MAX(" not in material
            and "ORDER BY" not in material
        )
    ):
        raise ValueError(
            "A pergunta pede o maior valor, "
            "mas o SQL não representa essa operação."
        )

    if (
        operacao == "min"
        and (
            "MIN(" not in material
            and "ORDER BY" not in material
        )
    ):
        raise ValueError(
            "A pergunta pede o menor valor, "
            "mas o SQL não representa essa operação."
        )



    # -----------------------------------------------------
    # 2. IDs resolvidos por uma etapa anterior
    # -----------------------------------------------------

    ids_resolvidos = (
        ids_resolvidos
        or []
    )

    for identificador in ids_resolvidos:

        if (
            "COMPONENTES" in material
            and identificador.upper()
            not in material
        ):
            raise ValueError(
                "O SQL não utiliza a entidade "
                f"já resolvida: {identificador}"
            )
# ============================================================
# Execução SQL controlada
# ============================================================

PROIBIDOS = {
    "insert",
    "update",
    "delete",
    "drop",
    "alter",
    "create",
    "pragma",
    "attach",
    "detach",
    "replace",
    "vacuum",
}


# Aceita um objeto ConsultaSQL ou converte uma string antiga nesse formato.
# Concentrar a compatibilidade aqui evita repetir essa verificação nos chamadores.
# isinstance verifica o tipo do objeto; valores de outros tipos geram TypeError.
def _consulta_sql(
    consulta: ConsultaSQL | str,
) -> ConsultaSQL:
    """
    Normaliza a entrada.

    O pipeline novo trabalha com ConsultaSQL,
    mas mantemos compatibilidade com células antigas
    que ainda passam uma string.
    """
    if isinstance(consulta, ConsultaSQL):
        return consulta

    if isinstance(consulta, str):
        return ConsultaSQL(
            sql=consulta,
            params=(),
            origem="string_direta",
        )

    raise TypeError(
        "A consulta deve ser str ou ConsultaSQL."
    )


# Aplica restrições básicas de consulta única iniciada por SELECT.
# Remover o ponto e vírgula final permite aceitar a terminação usual do SQL.
# Palavras proibidas são checadas por texto, inclusive em literais; este validador
# é simplificado e não faz uma análise completa da estrutura SQL.
def validar_sql(
    consulta: ConsultaSQL | str,
) -> None:
    """
    Guardrails básicos antes da execução.

    - somente SELECT;
    - nenhuma operação de escrita/DDL;
    - apenas uma instrução.
    """
    consulta = _consulta_sql(consulta)

    sql = consulta.sql.strip()

    if not sql:
        raise ValueError(
            "Consulta SQL vazia."
        )

    # Retiramos apenas o ; final para validar
    sql_sem_final = sql.rstrip(";").strip()

    sql_normalizado = normalizar(
        sql_sem_final
    )

    # ---------------------------------------
    # 1. Somente SELECT
    # ---------------------------------------

    if not sql_normalizado.startswith(
        "select"
    ):
        raise ValueError(
            "Somente SELECT é permitido."
        )

    # ---------------------------------------
    # 2. Palavras proibidas
    # ---------------------------------------

    for palavra in PROIBIDOS:
        if re.search(
            rf"\b{palavra}\b",
            sql_normalizado,
        ):
            raise ValueError(
                f"Operação proibida: {palavra}"
            )

    # ---------------------------------------
    # 3. Uma única instrução
    # ---------------------------------------

    if ";" in sql_sem_final:
        raise ValueError(
            "Somente uma instrução SQL é permitida."
        )


# Procura IDs esperados no SQL e nos parâmetros de consultas sobre componentes.
# A checagem detecta ausência evidente dos IDs, mas não prova que os filtros
# restringem o resultado exatamente a eles. A inspeção da consulta continua útil.
def validar_sql_respeita_ids(
    consulta: ConsultaSQL | str,
    ids: list[str],
) -> None:
    """
    Guardrail da rota vetor + SQL.

    Se uma etapa anterior resolveu determinadas
    entidades, o SQL não pode ignorá-las.

    Consideramos tanto o texto do SQL quanto
    os parâmetros, pois os templates usam ?.
    """
    consulta = _consulta_sql(
        consulta
    )

    if not ids:
        return

    material = (
        consulta.sql.upper()
        + " "
        + " ".join(
            str(valor).upper()
            for valor in consulta.params
        )
    )

    # Só aplicamos este guardrail quando
    # a consulta trabalha com componentes.
    if (
        "FROM COMPONENTES" not in material
        and "JOIN COMPONENTES" not in material
    ):
        return

    faltantes = [
        identificador
        for identificador in ids
        if identificador.upper()
        not in material
    ]

    if faltantes:
        raise ValueError(
            "O SQL não utiliza as entidades "
            "resolvidas anteriormente: "
            f"{faltantes}"
        )


# Executa a consulta validada e devolve dados estruturados mais evidências.
# O cursor fornece nomes de colunas e linhas; zip associa cada nome a seu valor.
# Buscamos uma linha a mais para sinalizar truncamento sem carregar todo o resultado.
# Esse limite reduz os dados retornados, não necessariamente o trabalho interno do SQLite.
def executar_sql(
    consulta: ConsultaSQL | str,
    perfil: str = "publico",
    limite_linhas: int = 100,
) -> tuple[dict, list[Evidencia]]:
    """
    Executa uma consulta SQL de forma controlada.

    Etapas:
    1. normaliza ConsultaSQL/string;
    2. valida perfil;
    3. valida SQL;
    4. abre SQLite somente leitura;
    5. gera EXPLAIN QUERY PLAN;
    6. executa com parâmetros;
    7. limita o resultado;
    8. transforma as linhas em evidências.
    """

    consulta = _consulta_sql(
        consulta
    )

    # Garante que o perfil existe.
    # As regras específicas de escopo já são
    # aplicadas pelo pipeline e pela resolução
    # das entidades.
    escopos_permitidos(
        perfil
    )

    validar_sql(
        consulta
    )

    sql_execucao = (
        consulta.sql
        .strip()
        .rstrip(";")
        .strip()
    )

    if not CAMINHO_SQLITE.exists():
        raise FileNotFoundError(
            f"Banco não encontrado em: "
            f"{CAMINHO_SQLITE}"
        )

    conn = sqlite3.connect(
        (
            f"file:"
            f"{CAMINHO_SQLITE.resolve()}"
            f"?mode=ro"
        ),
        uri=True,
    )

    try:

        # Reforça que a conexão é somente leitura.
        conn.execute(
            "PRAGMA query_only = ON"
        )

        # ----------------------------------------
        # Plano de execução
        # ----------------------------------------

        # EXPLAIN QUERY PLAN descreve a estratégia do banco (por exemplo, busca por índice).
        # Ele verifica aspectos da execução, mas não decide se o SQL corresponde à pergunta.
        plano = conn.execute(
            (
                "EXPLAIN QUERY PLAN "
                + sql_execucao
            ),
            consulta.params,
        ).fetchall()

        # ----------------------------------------
        # Execução
        # ----------------------------------------

        cursor = conn.execute(
            sql_execucao,
            consulta.params,
        )

        colunas = [
            coluna[0]
            for coluna in cursor.description
        ]

        # Pegamos uma linha adicional para
        # sabermos se houve truncamento.
        # Com limite 100, buscamos 101 linhas. A 101ª serve só para detectar
        # que existe resultado além do limite; ela não entra na lista final de evidências.
        linhas_brutas = cursor.fetchmany(
            limite_linhas + 1
        )

        truncado = (
            len(linhas_brutas)
            > limite_linhas
        )

        linhas = linhas_brutas[
            :limite_linhas
        ]

    finally:
        conn.close()

    # --------------------------------------------
    # Resultado estruturado
    # --------------------------------------------

    resultado = {
        "sql": sql_execucao,
        "params": consulta.params,
        "origem": consulta.origem,
        "explain": plano,
        "colunas": colunas,
        "linhas": linhas,
        "truncado": truncado,
    }

    # --------------------------------------------
    # Evidências
    # --------------------------------------------

    evidencias = []

    # Cada linha é uma tupla de valores. zip(colunas, linha) forma pares nome/valor
    # que viram um texto legível, mantendo também SQL e parâmetros nos metadados.
    for linha in linhas:

        conteudo = "; ".join(
            (
                f"{coluna}={valor}"
            )
            for coluna, valor
            in zip(
                colunas,
                linha,
            )
        )

        evidencias.append(
            Evidencia(
                tipo="sql",
                conteudo=conteudo,
                fonte="academico.db",
                metadata={
                    "sql": sql_execucao,
                    "params": list(
                        consulta.params
                    ),
                    "origem": consulta.origem,
                    "explain": plano,
                    "perfil": perfil,
                },
            )
        )

    return (
        resultado,
        evidencias,
    )

### Exemplo NL2SQL isolado

In [25]:
# Objetivo: acompanhar tradução e execução de SQL fora do pipeline completo.
# nl2sql devolve um objeto ConsultaSQL; executar_sql devolve resultado e evidências.
# EXPLAIN descreve como o SQLite planeja buscar os dados, não se a pergunta foi bem interpretada.
# Esta célula chama a validação básica de executar_sql; não chama separadamente
# validar_intencao_sql. Compare o SQL exibido com a soma solicitada.

pergunta = (
    "Qual é a carga horária "
    "total do TCC1 e do TCC2?"
)

sql = nl2sql(
    pergunta
)

resultado, evidencias = (
    executar_sql(
        sql
    )
)

print("SQL GERADO:")
print(sql)

print("\nEXPLAIN:")
print(
    resultado["explain"]
)

print("\nRESULTADO:")
print(
    resultado["linhas"]
)

SQL GERADO:
SELECT SUM(carga_horaria) AS total_carga_horaria FROM componentes WHERE nome IN ('Trabalho de Conclusão de Curso 1', 'Trabalho de Conclusão de Curso 2')

EXPLAIN:
[(3, 0, 0, 'SCAN componentes')]

RESULTADO:
[(300,)]


# PARTE VII — NL2Graph

## 18. Grafo de conhecimento mínimo para recuperação por relações

O grafo representa **entidades** e **relações explícitas** entre elas.

Ele é usado quando a pergunta exige percorrer relações como pré-requisito, referência ou dependência. A similaridade vetorial pode encontrar trechos relacionados, mas não garante a semântica nem a direção da relação.

> **Escopo:** aqui usamos um grafo de conhecimento simples para recuperação por relações. Isso não constitui, por si só, uma implementação completa de GraphRAG.

In [ ]:
# Objetivo: representar entidades e relações explícitas entre elas.
# DiGraph cria um grafo direcionado: A -> B tem significado diferente de B -> A.
# Os nós e as relações são cadastrados manualmente para esta demonstração.
# Fonte identifica a origem declarada da relação; o código não verifica o PDF ao cadastrar.
# O grafo é mantido em memória: reexecutar esta célula o constrói novamente.
# Para pré-requisitos usamos requisito -> disciplina, por isso a consulta busca entradas.

grafo = nx.DiGraph()

# ------------------------------------------------------------
# Nós: entidades + metadados de escopo
# ------------------------------------------------------------

grafo.add_node(
    "CI1218",
    nome="Bancos de Dados",
    tipo="disciplina",
    escopo="disciplina",
)

grafo.add_node(
    "PPC",
    nome="Projeto Pedagógico do Curso",
    tipo="documento",
    escopo="curso",
)

grafo.add_node(
    "ESTAGIO",
    nome="Estágio Obrigatório",
    tipo="atividade",
    escopo="estagio",
)

grafo.add_node(
    "REGULAMENTO_ESTAGIO",
    nome="Regulamento de Estágio",
    tipo="documento",
    escopo="estagio",
)

grafo.add_node(
    "PET",
    nome="PET",
    tipo="atividade",
    escopo="estagio",
)

grafo.add_node(
    "IC",
    nome="Iniciação Científica",
    tipo="atividade",
    escopo="estagio",
)

grafo.add_node(
    "TRABALHO_EMPRESA",
    nome="Trabalho em Empresa",
    tipo="atividade",
    escopo="estagio",
)

grafo.add_node(
    "DISCIPLINAS_BASICAS",
    nome="Disciplinas Básicas",
    tipo="grupo",
    escopo="curso",
)


# ------------------------------------------------------------
# Arestas: relação explícita + proveniência
# ------------------------------------------------------------

# Registra uma aresta direcionada com tipo e fonte.
# Uma função centraliza os nomes dos atributos para todas as relações.
# DiGraph admite uma aresta por par ordenado; adicionar outra no mesmo par
# atualiza atributos em vez de manter múltiplas relações independentes.
def adicionar_relacao(
    origem: str,
    destino: str,
    relacao: str,
    fonte: str,
) -> None:
    grafo.add_edge(
        origem,
        destino,
        relacao=relacao,
        fonte=fonte,
    )


adicionar_relacao(
    "PPC",
    "ESTAGIO",
    "preve",
    "PPC",
)

adicionar_relacao(
    "REGULAMENTO_ESTAGIO",
    "PPC",
    "referencia",
    "Regulamento de Estágio",
)

adicionar_relacao(
    "ESTAGIO",
    "DISCIPLINAS_BASICAS",
    "exige_conclusao",
    "PPC + Regulamento",
)

adicionar_relacao(
    "PET",
    "ESTAGIO",
    "pode_validar_como",
    "Regulamento, Art. 29",
)

adicionar_relacao(
    "IC",
    "ESTAGIO",
    "pode_validar_como",
    "Regulamento",
)

adicionar_relacao(
    "TRABALHO_EMPRESA",
    "ESTAGIO",
    "pode_substituir_horas",
    "Regulamento",
)


# ------------------------------------------------------------
# Pré-requisitos DIRETOS cadastrados para CI1218
# ------------------------------------------------------------

# Esta lista é o cadastro usado no exercício, não uma extração feita nesta célula.
# Revise os códigos na fonte citada antes de tratá-los como requisitos oficiais.
# O laço abaixo cria uma relação direta de cada código para CI1218.
prerequisitos = [
    "CI1055",
    "CI1068",
    "CI1003",
    "CMA111",
    "CM304",
    "CI1056",
    "CI1210",
    "CI1001",
    "CMA211",
    "CM303",
    "CI1057",
    "CI1212",
    "CI1002",
    "CI1237",
    "CE009",
]

for codigo in prerequisitos:
    grafo.add_node(
        codigo,
        nome=codigo,
        tipo="disciplina",
        escopo="disciplina",
    )

    adicionar_relacao(
        codigo,
        "CI1218",
        "pre_requisito_de",
        "Ficha CI1218",
    )

print(
    grafo.number_of_nodes(),
    "nós |",
    grafo.number_of_edges(),
    "arestas",
)


## 19. Linguagem natural -> consulta no grafo

A pergunta é traduzida para uma estrutura controlada com **entidade**, **relação**, **direção** e **profundidade**.

- `profundidade = 1`: somente relações diretas;
- `profundidade = null`: percurso transitivo até onde houver relações compatíveis.

In [ ]:
# Objetivo: traduzir a pergunta em instruções limitadas para percorrer o grafo.
# Em vez de gerar código Python, o modelo propõe um dicionário com entidade,
# relações, direção e profundidade. O programa valida esses campos antes de usá-los.
# Perguntas diretas têm regras próprias; o modelo é uma alternativa para os outros casos.
# Profundidade 1 significa uma aresta; None permite continuar pelos nós alcançáveis.

RELACOES_PERMITIDAS = {
    "pre_requisito_de",
    "preve",
    "referencia",
    "exige_conclusao",
    "pode_validar_como",
    "pode_substituir_horas",
}


# Procura IDs e nomes conhecidos dentro da pergunta normalizada.
# Entre correspondências, prefere o nome mais longo por ser geralmente mais específico.
# Isso reduz correspondências genéricas, mas continua sendo uma regra textual.
def resolver_entidade_grafo(
    pergunta: str,
) -> str | None:
    """
    Resolve entidades conhecidas a partir do ID ou do nome do nó.

    Evita regras específicas como:
        "bancos de dados" -> CI1218
    espalhadas pelo roteador.
    """
    p = normalizar(pergunta)
    candidatos = []

    for entidade, dados in grafo.nodes(data=True):
        aliases = [
            entidade,
            dados.get("nome", ""),
        ]

        for alias in aliases:
            alias_norm = normalizar(
                str(alias)
            ).strip()

            if (
                alias_norm
                and alias_norm in p
            ):
                candidatos.append(
                    (
                        len(alias_norm),
                        entidade,
                    )
                )

    if not candidatos:
        return None

    # Prefere o alias mais específico encontrado.
    # Cada candidato é (comprimento do nome, ID). Tuplas são comparadas por posição.
    # reverse=True prioriza nomes maiores; em empate o ID também influencia a ordem.
    candidatos.sort(
        reverse=True
    )

    return candidatos[0][1]


# Produz consultas de grafo para padrões conhecidos sem chamar o modelo.
# Pré-requisitos usam entrada porque a aresta parte do requisito e chega à disciplina.
# Usar profundidade 1 distingue dependências diretas de percursos transitivos.
def graph_por_regras(
    pergunta: str,
) -> dict | None:
    """
    Resolve relações muito claras antes de consultar o LLM.
    """
    p = normalizar(pergunta)
    entidade = resolver_entidade_grafo(
        pergunta
    )

    # --------------------------------------------------------
    # Pré-requisito / antecedência
    # --------------------------------------------------------

    pergunta_prerequisito = any(
        termo in p
        for termo in [
            "pre requisito",
            "pre requisitos",
            "prerequisito",
            "prerequisitos",
            "antecede",
            "antecedem",
            "antecedente",
            "antecedentes",
        ]
    )

    if (
        pergunta_prerequisito
        and entidade is not None
    ):
        # "pré-requisitos" -> diretos por padrão.
        # "antecedem" ou menção explícita a indiretos -> transitivo.
        transitivo = any(
            termo in p
            for termo in [
                "indireto",
                "indiretos",
                "indiretamente",
                "direta ou indiretamente",
                "antecede",
                "antecedem",
                "antecedente",
                "antecedentes",
            ]
        )

        return {
            "entidade": entidade,
            "relacoes": ["pre_requisito_de"],
            "direcao": "entrada",
            "profundidade": None if transitivo else 1,
            "origem_decisao": "regra",
        }

    # --------------------------------------------------------
    # Atividades relacionadas a estágio
    # --------------------------------------------------------

    if (
        entidade == "ESTAGIO"
        and any(
            termo in p
            for termo in [
                "validar",
                "validado",
                "validada",
                "substituir",
                "substituicao",
                "atividade",
                "atividades",
            ]
        )
    ):
        return {
            "entidade": "ESTAGIO",
            "relacoes": [
                "pode_validar_como",
                "pode_substituir_horas",
            ],
            "direcao": "entrada",
            "profundidade": 1,
            "origem_decisao": "regra",
        }

    # --------------------------------------------------------
    # PPC -> estágio
    # --------------------------------------------------------

    if (
        entidade == "PPC"
        and (
            "preve" in p
            or "relação" in p
            or "relacao" in p
        )
    ):
        return {
            "entidade": "PPC",
            "relacoes": ["preve"],
            "direcao": "saida",
            "profundidade": 1,
            "origem_decisao": "regra",
        }

    return None


# Converte variações de formato e verifica entidade, relações e direção.
# Uma string de relação vira lista para o restante do código tratar um único formato.
# Erros são informados antes do percurso para não aceitar nós/relações inventados.
def normalizar_consulta_grafo(
    consulta: dict,
) -> dict:
    """
    Normaliza e valida a estrutura produzida por regra ou LLM.
    """
    entidade = consulta.get(
        "entidade"
    )

    direcao = consulta.get(
        "direcao"
    )

    relacoes = consulta.get(
        "relacoes"
    )

    if relacoes is None:
        relacao = consulta.get(
            "relacao"
        )
        relacoes = (
            [relacao]
            if relacao
            else []
        )

    # Uma única string viraria caracteres se iterada diretamente.
    # Envolvê-la em [relacoes] produz uma lista de um item para o laço seguinte.
    if isinstance(relacoes, str):
        relacoes = [relacoes]

    relacoes_invalidas = [
        relacao
        for relacao in relacoes
        if relacao not in RELACOES_PERMITIDAS
    ]

    if relacoes_invalidas:
        raise ValueError(
            "Relações não permitidas: "
            f"{relacoes_invalidas}"
        )

    if entidade not in grafo:
        raise ValueError(
            f"Entidade não existe no grafo: {entidade}"
        )

    if not relacoes:
        raise ValueError(
            "Nenhuma relação válida foi produzida."
        )

    if direcao not in {
        "entrada",
        "saida",
    }:
        raise ValueError(
            "Direção inválida."
        )

    profundidade = consulta.get(
        "profundidade",
        1,
    )

    if profundidade is not None:
        if (
            not isinstance(profundidade, int)
            or profundidade < 1
        ):
            raise ValueError(
                "profundidade deve ser inteiro >= 1 ou null."
            )

    return {
        "entidade": entidade,
        "relacoes": list(relacoes),
        "direcao": direcao,
        "profundidade": profundidade,
        "origem_decisao": consulta.get(
            "origem_decisao",
            "llm",
        ),
    }


# Traduz a pergunta em um plano de consulta por regras ou pelo LLM.
# Fornece ao modelo nós e relações existentes e valida a estrutura produzida.
# O JSON retornado é dado; não é executado como código Python.
def nl2graph(
    pergunta: str,
) -> dict:
    """
    Traduz linguagem natural para uma consulta controlada ao grafo.
    """
    consulta_regra = graph_por_regras(
        pergunta
    )

    if consulta_regra is not None:
        return normalizar_consulta_grafo(
            consulta_regra
        )

    nos_conhecidos = "\n".join(
        f"- {entidade}: {dados.get('nome', entidade)}"
        for entidade, dados
        in grafo.nodes(data=True)
    )

    relacoes_conhecidas = "\n".join(
        f"- {relacao}"
        for relacao
        in sorted(RELACOES_PERMITIDAS)
    )

    system = f"""
Você traduz perguntas para consultas em um grafo de conhecimento.

Retorne APENAS JSON no formato:
{{
  "entidade": "...",
  "relacoes": ["..."],
  "direcao": "entrada" ou "saida",
  "profundidade": 1 ou null
}}

Semântica de profundidade:
- 1 = apenas relações diretas;
- null = percurso transitivo pelas relações solicitadas.

Nós conhecidos:
{nos_conhecidos}

Relações permitidas:
{relacoes_conhecidas}

Exemplo 1:
Pergunta: Quais são os pré-requisitos diretos da CI1218?
Saída:
{{"entidade":"CI1218","relacoes":["pre_requisito_de"],"direcao":"entrada","profundidade":1}}

Exemplo 2:
Pergunta: Quais disciplinas antecedem a CI1218 direta ou indiretamente?
Saída:
{{"entidade":"CI1218","relacoes":["pre_requisito_de"],"direcao":"entrada","profundidade":null}}

Não responda à pergunta. Gere somente a consulta.
""".strip()

    consulta = extrair_json(
        chamar_llm(
            system,
            pergunta,
            max_new_tokens=180,
        )
    )

    consulta["origem_decisao"] = "llm"

    return normalizar_consulta_grafo(
        consulta
    )


## 20. Execução controlada do grafo

A execução respeita **tipo da relação**, **direção**, **profundidade** e **permissões**. O resultado inclui o caminho percorrido e a proveniência das arestas.

In [ ]:
# Objetivo: executar a consulta usando apenas relações cadastradas no grafo.
# Uma fila organiza a exploração por distância: primeiro vizinhos diretos, depois
# os vizinhos deles. Essa estratégia é chamada busca em largura.
# Um conjunto de visitados impede expandir repetidamente o mesmo nó em ciclos.
# O resultado traz caminhos e fontes; ele depende da qualidade das relações cadastradas.

# Confere se o escopo do nó pertence ao perfil.
# Esta implementação também permite nós sem escopo; None não é bloqueado aqui.
# Por isso cadastrar metadados de escopo nos nós faz parte da política demonstrada.
def _entidade_autorizada_grafo(
    entidade: str,
    perfil: str,
) -> bool:
    permitidos = escopos_permitidos(
        perfil
    )

    escopo = grafo.nodes[
        entidade
    ].get("escopo")

    return (
        escopo is None
        or escopo in permitidos
    )


# Impede iniciar o percurso por uma entidade fora do perfil.
# Os vizinhos serão verificados novamente durante a exploração.
# Separar as duas verificações cobre tanto o ponto de partida quanto os nós alcançados.
def validar_permissao_grafo(
    consulta: dict,
    perfil: str = "publico",
) -> None:
    """
    A entidade inicial precisa estar autorizada.
    """
    entidade = consulta[
        "entidade"
    ]

    if not _entidade_autorizada_grafo(
        entidade,
        perfil,
    ):
        escopo = grafo.nodes[
            entidade
        ].get("escopo")

        raise PermissionError(
            f"Perfil '{perfil}' não pode consultar "
            f"a entidade '{entidade}' do escopo '{escopo}'."
        )


# Produz somente arestas da direção e dos tipos solicitados.
# yield devolve um item por vez e pausa a função até o próximo passo do laço consumidor.
# Esse gerador evita montar uma lista intermediária de todas as arestas compatíveis.
def _arestas_compativeis(
    entidade: str,
    relacoes: set[str],
    direcao: str,
):
    """
    Produz arestas compatíveis com relação e direção.

    Retorna:
        vizinho, origem, destino, dados
    """
    if direcao == "entrada":
        for origem in grafo.predecessors(
            entidade
        ):
            dados = grafo.get_edge_data(
                origem,
                entidade,
            )

            if dados.get("relacao") in relacoes:
                yield (
                    origem,
                    origem,
                    entidade,
                    dados,
                )

    else:
        for destino in grafo.successors(
            entidade
        ):
            dados = grafo.get_edge_data(
                entidade,
                destino,
            )

            if dados.get("relacao") in relacoes:
                yield (
                    destino,
                    entidade,
                    destino,
                    dados,
                )


# Transforma nós e arestas em uma sequência textual com setas.
# zip emparelha cada relação com o próximo nó do caminho.
# O texto preserva direção e tipo para que a resposta possa citar a relação correta.
def _formatar_caminho_grafo(
    nos: list[str],
    arestas: list[dict],
) -> str:
    partes = [
        nos[0]
    ]

    for dados, destino in zip(
        arestas,
        nos[1:],
    ):
        partes.append(
            f" --{dados['relacao']}--> "
        )
        partes.append(
            destino
        )

    return "".join(
        partes
    )


# Explora relações válidas e monta evidências dos caminhos encontrados.
# Usa fila e conjunto de visitados para controlar a exploração e evitar ciclos.
# Cada nó é expandido no máximo uma vez; o algoritmo não enumera todos os caminhos
# possíveis de um grafo com múltiplas formas de chegar à mesma entidade.
def executar_grafo(
    consulta: dict,
    perfil: str = "publico",
) -> list[Evidencia]:
    """
    Executa a consulta sem inferir relações inexistentes.

    profundidade = 1
        somente vizinhos diretos.

    profundidade = None
        percurso transitivo por relações compatíveis.
    """
    consulta = normalizar_consulta_grafo(
        consulta
    )

    validar_permissao_grafo(
        consulta,
        perfil=perfil,
    )

    entidade_alvo = consulta[
        "entidade"
    ]

    relacoes = set(
        consulta["relacoes"]
    )

    direcao = consulta[
        "direcao"
    ]

    profundidade_max = consulta[
        "profundidade"
    ]

    evidencias = []

    # Cada item:
    # (nó atual, nós do caminho, arestas do caminho, profundidade)
    fila = [
        (
            entidade_alvo,
            [entidade_alvo],
            [],
            0,
        )
    ]

    visitados = {
        entidade_alvo
    }

    # while repete enquanto a lista não estiver vazia.
    # pop(0) retira o primeiro item e as expansões entram no fim, mantendo ordem por nível.
    # Uma lista é simples para esta demonstração; em grafos grandes, deque.popleft()
    # evitaria o custo de deslocar os itens restantes a cada remoção.
    while fila:
        (
            atual,
            nos_caminho,
            arestas_caminho,
            profundidade_atual,
        ) = fila.pop(0)

        if (
            profundidade_max is not None
            and profundidade_atual >= profundidade_max
        ):
            continue

        for (
            vizinho,
            origem,
            destino,
            dados,
        ) in _arestas_compativeis(
            atual,
            relacoes,
            direcao,
        ):
            # Não expõe entidades fora do escopo do perfil.
            if not _entidade_autorizada_grafo(
                vizinho,
                perfil,
            ):
                continue

            nova_profundidade = (
                profundidade_atual + 1
            )

            # Ao seguir entradas, encontramos nós anteriores ao alvo.
            # Por isso o vizinho é colocado no início do caminho para as setas preservarem
            # a direção original da relação, mesmo percorrendo-a no sentido inverso.
            if direcao == "entrada":
                novos_nos = (
                    [vizinho]
                    + nos_caminho
                )
                novas_arestas = (
                    [dados]
                    + arestas_caminho
                )
            else:
                novos_nos = (
                    nos_caminho
                    + [vizinho]
                )
                novas_arestas = (
                    arestas_caminho
                    + [dados]
                )

            caminho = _formatar_caminho_grafo(
                novos_nos,
                novas_arestas,
            )

            fontes = list(
                dict.fromkeys(
                    aresta.get(
                        "fonte",
                        "grafo",
                    )
                    for aresta
                    in novas_arestas
                )
            )

            evidencias.append(
                Evidencia(
                    tipo="grafo",
                    conteudo=caminho,
                    fonte=" | ".join(
                        fontes
                    ),
                    metadata={
                        **consulta,
                        "profundidade_encontrada": nova_profundidade,
                        "nos_caminho": novos_nos,
                        "relacoes_caminho": [
                            aresta["relacao"]
                            for aresta
                            in novas_arestas
                        ],
                    },
                )
            )

            # Em grafos com ciclos, não revisitamos o mesmo nó.
            # A evidência da aresta já foi registrada; visitados controla apenas a expansão.
            # Assim evitamos laços infinitos, mas ainda podemos registrar uma aresta para um nó visto.
            if vizinho not in visitados:
                visitados.add(
                    vizinho
                )

                fila.append(
                    (
                        vizinho,
                        novos_nos,
                        novas_arestas,
                        nova_profundidade,
                    )
                )

    if not evidencias:
        raise ValueError(
            "A consulta de grafo foi válida, "
            "mas não encontrou relações correspondentes."
        )

    return evidencias


### Exemplo NL2Graph isolado — relação direta

A pergunta abaixo pede os **pré-requisitos diretos**. Para relações transitivas, use uma pergunta como “Quais disciplinas antecedem a CI1218 direta ou indiretamente?”.

In [ ]:
# Objetivo: conferir uma consulta de pré-requisitos diretos antes da resposta final.
# nl2graph gera o plano; executar_grafo percorre as relações e devolve evidências.
# Confira direção='entrada' e profundidade=1: isso corresponde a requisito -> CI1218.
# O laço imprime cada caminho e a fonte cadastrada para facilitar a revisão.

pergunta = (
    "Quais são os pré-requisitos diretos "
    "da CI1218?"
)

consulta = nl2graph(
    pergunta
)

evidencias = executar_grafo(
    consulta
)

print(
    json.dumps(
        consulta,
        ensure_ascii=False,
        indent=2,
    )
)

for evidencia in evidencias:
    print(
        "-",
        evidencia.conteudo,
        "| fonte:",
        evidencia.fonte,
    )


# PARTE VIII — Rotas híbridas

## 21. Vetor + SQL

Exemplo:

> “Qual é a carga horária da disciplina que trata de processamento de consultas e otimização?”

A pergunta não informa `CI1218`.

1. vetor identifica semanticamente a disciplina;
2. LLM extrai `CI1218`;
3. NL2SQL consulta o valor exato.

> **Ajuste importante:** nessa rota há dois pontos adicionais de falha: a resolução da entidade e a tradução NL2SQL. A entidade é resolvida primeiro por fonte/metadados e o SQL é obrigado a usar o ID encontrado.

In [ ]:
# Objetivo: ligar a descrição textual encontrada no PDF ao ID usado no SQLite.
# A busca semântica pode encontrar tanto o curso quanto uma disciplina; os candidatos
# ainda serão refinados pelo tipo solicitado e pelas permissões em nl2sql_com_ids.
# Usamos regras baseadas nas evidências antes do LLM para reaproveitar identificadores
# já visíveis no texto ou na fonte. A lista de IDs permitidos é específica deste corpus.

IDS_PERMITIDOS = {
    "CI1218",
    "ESTAGIO",
    "CURSO",
    "TCC1",
    "TCC2",
    "ATIVIDADES_FORMATIVAS",
}


# Extrai IDs candidatos dos textos e metadados das evidências.
# Usar arquivo e trecho original ajuda quando a compressão removeu o nome da entidade.
# As regras são específicas dos documentos da aula; novos domínios exigiriam outros mapeamentos.
def identificar_ids_deterministicamente(
    evidencias: list[Evidencia],
) -> list[str]:
    """
    Primeiro tenta resolver a entidade a partir da própria evidência.

    Isso é especialmente importante em vetor+SQL:
    se a busca recuperou ci1218.pdf, não precisamos pedir
    ao LLM pequeno para "redescobrir" CI1218.
    """

    encontrados = []

    for evidencia in evidencias:
        alvo = normalizar(
            " ".join([
                evidencia.fonte,
                evidencia.conteudo,
                str(
                    evidencia.metadata.get(
                        "arquivo",
                        "",
                    )
                ),
                str(
                    evidencia.metadata.get(
                        "texto_original",
                        "",
                    )
                ),
            ])
        )

        if (
            "ci1218" in alvo
            or "bancos de dados" in alvo
        ):
            encontrados.append(
                "CI1218"
            )

        if (
            "estagio obrigatorio" in alvo
            or "regulamento de estagio" in alvo
        ):
            encontrados.append(
                "ESTAGIO"
            )

        if (
            "projeto pedagogico" in alvo
            or " ppc " in f" {alvo} "
        ):
            encontrados.append(
                "CURSO"
            )

        if "tcc1" in alvo:
            encontrados.append(
                "TCC1"
            )

        if "tcc2" in alvo:
            encontrados.append(
                "TCC2"
            )

        if (
            "atividades formativas" in alvo
        ):
            encontrados.append(
                "ATIVIDADES_FORMATIVAS"
            )

    return list(
        dict.fromkeys(
            encontrados
        )
    )


# Reaproveita IDs encontrados por regras; só chama o modelo se não encontrar nenhum.
# O filtro IDS_PERMITIDOS restringe a saída do LLM a identificadores conhecidos.
# Uma lista vazia indica que esta etapa não resolveu a entidade, permitindo ao chamador
# interromper a consulta numérica em vez de escolher um ID arbitrariamente.
def identificar_ids_llm(
    pergunta: str,
    evidencias: list[Evidencia],
) -> list[str]:
    """
    Estratégia híbrida:
    1. identificação determinística a partir de fonte/metadados;
    2. LLM somente se nenhuma entidade segura foi encontrada.
    """

    ids_regra = (
        identificar_ids_deterministicamente(
            evidencias
        )
    )

    # Uma lista não vazia vale True. Quando as regras já produziram candidatos,
    # retornamos todos eles; a filtragem por tipo solicitado acontece depois no SQL.
    if ids_regra:
        return ids_regra

    contexto = "\n\n".join(
        (
            f"FONTE: {e.fonte}\n"
            f"EVIDÊNCIA: {e.conteudo}\n"
            f"METADADOS: {e.metadata}"
        )
        for e in evidencias
    )

    system = """
Extraia somente IDs explicitamente sustentados pelas evidências.

IDs permitidos:
CI1218
ESTAGIO
CURSO
TCC1
TCC2
ATIVIDADES_FORMATIVAS

Retorne APENAS:
{"ids":["..."]}

Se não houver identificação segura:
{"ids":[]}
""".strip()

    try:
        obj = extrair_json(
            chamar_llm(
                system,
                (
                    f"PERGUNTA:\n{pergunta}\n\n"
                    f"EVIDÊNCIAS:\n{contexto}"
                ),
                max_new_tokens=100,
            )
        )

        return [
            item
            for item in obj.get(
                "ids",
                [],
            )
            if item in IDS_PERMITIDOS
        ]

    except Exception:
        return []

# PARTE IX — Pós-processamento comum

## 22. Evidências heterogêneas

Depois das rotas, podemos ter:

- chunk vetorial;
- resultado SQL;
- aresta do grafo.

Agora precisamos montar um contexto comum sem perder:
- tipo;
- fonte;
- metadados;
- consulta SQL;
- relação do grafo;
- versão.

In [ ]:
# Objetivo: apresentar evidências de formatos diferentes em um único texto.
# Todas recebem número, tipo, fonte e conteúdo, permitindo citações como [1] e [2].
# O limite é em caracteres, não tokens. Ele controla o tamanho aproximado do contexto,
# mas não substitui uma contagem exata de tokens do modelo.
# A ordem recebida importa porque paramos ao atingir o limite.

LIMITE_CARACTERES_CONTEXTO = 6500


# Numera as evidências e monta o texto a ser enviado ao modelo.
# Acumular blocos em uma lista facilita acompanhar o limite antes de juntar as strings.
# break para no primeiro bloco que não cabe; evidências seguintes também ficam de fora.
def formatar_evidencias(
    evidencias: list[Evidencia],
    limite: int = LIMITE_CARACTERES_CONTEXTO,
) -> str:
    blocos = []
    usados = 0

    for indice, evidencia in enumerate(
        evidencias,
        start=1,
    ):
        bloco = (
            f"[{indice}]\n"
            f"TIPO: {evidencia.tipo}\n"
            f"FONTE: {evidencia.fonte}\n"
            f"EVIDÊNCIA: {evidencia.conteudo}\n"
        )

        if (
            usados + len(bloco)
            > limite
        ):
            # Interrompemos o laço em vez de cortar uma evidência no meio.
            # Isso preserva blocos completos, mas pode deixar de fora um bloco posterior menor.
            break

        blocos.append(
            bloco
        )
        usados += len(
            bloco
        )

    return "\n".join(
        blocos
    )

## 23. Contexto suficiente?

A aula anterior já discutia “contexto suficiente”.


Usaremos o LLM apenas para avaliar a suficiência — sem gerar a resposta ainda.

> **Ajuste:** a suficiência agora é híbrida. Casos claros são decididos por regras observáveis (cobertura dos termos centrais, similaridade, existência de resultado SQL/grafo). O LLM só é consultado quando o caso é realmente duvidoso.

> **Importante:** se SQL ou grafo não produzem nenhuma evidência, o problema não é 'contexto insuficiente' propriamente dito; é uma falha anterior de tradução/execução.


## Funções auxiliares para avaliar suficiência vetorial

Estas funções precisam ser definidas antes de `avaliar_suficiencia()`. Elas extraem os termos centrais da pergunta e montam o texto completo das evidências.

In [ ]:
# Objetivo: preparar textos e palavras importantes para avaliar a recuperação vetorial.
# Removemos palavras genéricas para não considerar 'qual' ou 'disciplina' como prova
# de que o trecho contém a informação procurada.
# Também incluímos o texto original disponível, pois a compressão pode omitir uma frase útil.
# Essas funções medem presença de termos; não verificam a verdade das afirmações.

TERMOS_GENERICOS_SUFICIENCIA = {
    "qual", "quais", "quanto", "quantos",
    "o", "a", "os", "as",
    "de", "da", "do", "das", "dos",
    "em", "no", "na", "nos", "nas",
    "sobre", "aborda", "trata", "fala",
    "disciplina", "curso", "documento",
    "conteudo", "conteudos",
}


# Seleciona termos para medir cobertura do assunto nos trechos recuperados.
# A compreensão de conjunto remove repetições automaticamente.
# Não há análise de sinônimos aqui: essa parte usa normalização e regras textuais.
def termos_centrais_pergunta(
    pergunta: str,
) -> set[str]:
    """
    Mantém apenas os termos informativos da pergunta.

    Exemplo:
    'O que a disciplina de Bancos de Dados aborda sobre
    processamento de consultas?'

    tende a manter:
    {'bancos', 'dados', 'processamento', 'consultas'}
    """
    return {
        termo
        for termo in normalizar(pergunta).split()
        if (
            len(termo) > 2
            and termo not in TERMOS_GENERICOS_SUFICIENCIA
        )
    }


# Combina conteúdo comprimido e texto original disponível para avaliar cobertura.
# Recuperar palavras omitidas pela compressão reduz falsos negativos nessa medição.
# O resultado é normalizado para comparar com os termos da pergunta.
def texto_total_evidencias(
    evidencias: list[Evidencia],
) -> str:
    """
    Junta o trecho comprimido e, quando disponível,
    o chunk original preservado nos metadados.

    Isso evita um falso negativo de suficiência quando
    a compressão remove justamente uma frase relevante.
    """
    partes = []

    for evidencia in evidencias:
        partes.append(
            str(evidencia.conteudo)
        )

        texto_original = evidencia.metadata.get(
            "texto_original"
        )

        if texto_original:
            partes.append(
                str(texto_original)
            )

    return normalizar(
        " ".join(partes)
    )

In [ ]:
# Objetivo: decidir se o material recuperado permite tentar uma resposta.
# Cada rota usa um critério: tipos de evidência para SQL/grafo e sinais textuais para vetor.
# As rotas híbridas exigem os dois tipos porque precisam ligar significado a dado/relação.
# Cobertura mede presença de termos, não correção factual. Os limiares são heurísticas.
# Uma decisão 'suficiente' permite gerar a resposta; a revisão humana continua necessária.

# Devolve suficiente, motivo e método para o pipeline decidir se responde.
# SQL/grafo são aceitos pela presença do tipo esperado; vetor usa cobertura e similaridade.
# Só casos vetoriais que não passam na regra vão ao avaliador LLM.
# Esses critérios indicam material disponível, não garantem a verdade da resposta futura.
def avaliar_suficiencia(
    pergunta: str,
    evidencias: list[Evidencia],
    rota: str,
) -> dict:
    """
    Avaliação de suficiência orientada pela ROTA.

    Ponto:
    - SQL/grafo já passaram por tradução, validação e execução;
    - se produziram evidência compatível com a rota, não faz sentido
      pedir a um LLM pequeno para "autorizar" novamente a resposta;
    - o LLM fica como avaliador apenas da rota puramente vetorial.
    """

    if not evidencias:
        return {
            "suficiente": False,
            "motivo": "nenhuma evidência foi produzida",
            "metodo": "regra_por_rota",
        }

    tipos = {
        evidencia.tipo
        for evidencia in evidencias
    }

    # --------------------------------------------------------
    # SQL
    # --------------------------------------------------------
    if rota == "sql":
        if "sql" in tipos:
            return {
                "suficiente": True,
                "motivo": (
                    "a rota SQL produziu resultado estruturado "
                    "após validação e execução controlada"
                ),
                "metodo": "regra_por_rota",
            }

        return {
            "suficiente": False,
            "motivo": "a rota SQL não produziu evidência SQL",
            "metodo": "regra_por_rota",
        }

    # --------------------------------------------------------
    # GRAFO
    # --------------------------------------------------------
    if rota == "grafo":
        if "grafo" in tipos:
            return {
                "suficiente": True,
                "motivo": (
                    "a rota de grafo produziu relações explícitas "
                    "após validação da consulta"
                ),
                "metodo": "regra_por_rota",
            }

        return {
            "suficiente": False,
            "motivo": "a rota de grafo não produziu relações",
            "metodo": "regra_por_rota",
        }

    # --------------------------------------------------------
    # VETOR + SQL
    # --------------------------------------------------------
    if rota == "vetor_sql":
        if "vetor" in tipos and "sql" in tipos:
            return {
                "suficiente": True,
                "motivo": (
                    "a busca vetorial identificou a entidade e "
                    "a etapa SQL produziu o valor estruturado"
                ),
                "metodo": "regra_por_rota",
            }

        return {
            "suficiente": False,
            "motivo": (
                "a rota vetor_sql exige evidência vetorial "
                "e evidência SQL"
            ),
            "metodo": "regra_por_rota",
        }

    # --------------------------------------------------------
    # VETOR + GRAFO
    # --------------------------------------------------------
    if rota == "vetor_grafo":
        if "vetor" in tipos and "grafo" in tipos:
            return {
                "suficiente": True,
                "motivo": (
                    "a rota vetor_grafo produziu evidência semântica "
                    "e relações explícitas"
                ),
                "metodo": "regra_por_rota",
            }

        return {
            "suficiente": False,
            "motivo": (
                "a rota vetor_grafo exige evidência vetorial "
                "e evidência de grafo"
            ),
            "metodo": "regra_por_rota",
        }

    # --------------------------------------------------------
    # VETOR PURO
    # --------------------------------------------------------
    if rota == "vetor":
        termos_pergunta = termos_centrais_pergunta(
            pergunta
        )

        texto_evidencias = texto_total_evidencias(
            evidencias
        )

        encontrados = {
            termo
            for termo in termos_pergunta
            if termo in texto_evidencias
        }

        # Exemplo: encontrar 3 de 5 termos centrais gera cobertura 0.6 (60%).
        # O "or 1" evita divisão por zero se não sobrou termo informativo.
        # Palavras presentes podem estar em contextos diferentes; cobertura não prova resposta.
        cobertura = (
            len(encontrados)
            / (len(termos_pergunta) or 1)
        )

        similaridades = [
            float(
                evidencia.metadata.get(
                    "similaridade",
                    0.0,
                )
            )
            for evidencia in evidencias
            if evidencia.tipo == "vetor"
        ]

        melhor_similaridade = max(
            similaridades,
            default=0.0,
        )

        # O limiar 60% é uma decisão didática para evitar chamadas adicionais ao LLM
        # quando há coincidência textual suficiente segundo a regra.
        # É diferente do fallback abaixo, que aceita 40% se a avaliação falhar.
        if (
            cobertura >= 0.60
            and melhor_similaridade >= LIMIAR_SIMILARIDADE
        ):
            return {
                "suficiente": True,
                "motivo": (
                    "as evidências vetoriais cobrem os conceitos "
                    f"centrais da pergunta ({cobertura:.0%})"
                ),
                "metodo": "regra_vetor",
                "cobertura": round(cobertura, 3),
                "termos_centrais": sorted(termos_pergunta),
                "termos_encontrados": sorted(encontrados),
                "melhor_similaridade": round(
                    melhor_similaridade,
                    4,
                ),
            }

        # Somente aqui usamos o LLM como fallback.
        contexto = formatar_evidencias(
            evidencias
        )

        system = """
Avalie se as evidências vetoriais são suficientes para responder.

- considere sinônimos e formulações equivalentes;
- não exija correspondência literal;
- não responda à pergunta;
- não use conhecimento externo.

Retorne APENAS JSON:
{
  "suficiente": true ou false,
  "motivo": "..."
}
""".strip()

        try:
            resultado = extrair_json(
                chamar_llm(
                    system,
                    (
                        f"PERGUNTA:\n{pergunta}\n\n"
                        f"EVIDÊNCIAS:\n{contexto}"
                    ),
                    max_new_tokens=120,
                )
            )

            resultado["suficiente"] = normalizar_bool(
                resultado.get("suficiente"),
                padrao=(cobertura >= 0.40),
            )

            resultado["metodo"] = "llm_fallback_vetor"
            resultado["cobertura"] = round(cobertura, 3)

            return resultado

        except Exception as erro:
            return {
                "suficiente": cobertura >= 0.40,
                "motivo": (
                    "fallback determinístico após falha "
                    "do avaliador vetorial"
                ),
                "metodo": "fallback_vetor",
                "cobertura": round(cobertura, 3),
                "erro_avaliador": str(erro),
            }

    return {
        "suficiente": False,
        "motivo": f"rota desconhecida para avaliação: {rota}",
        "metodo": "regra_por_rota",
    }

# Compatibilidade com células antigas do notebook.
# Prefira chamar avaliar_suficiencia(..., rota=...).
# Mantém compatibilidade com chamadas antigas, delegando à função atual.
# Um invólucro evita duplicar a lógica de avaliação em dois lugares.
# O nome histórico não significa que toda chamada realmente consulte o modelo.
def avaliar_suficiencia_llm(
    pergunta: str,
    evidencias: list[Evidencia],
    rota: str = "vetor",
) -> dict:
    return avaliar_suficiencia(
        pergunta,
        evidencias,
        rota=rota,
    )

## 24. Resposta final

In [ ]:
# Objetivo: redigir a resposta a partir do material recuperado nas etapas anteriores.
# O prompt pede referências numeradas e preservação de condições e números.
# Fornecer essas instruções reduz a liberdade da geração, mas não comprova que o modelo
# as cumpriu. Compare a resposta com as evidências ao revisar os testes.

# A resposta final é produzida somente depois que as evidências já foram
# recuperadas e validadas pelas rotas anteriores.
# Produz texto final usando o contexto numerado.
# Separar redação e recuperação permite revisar as fontes antes de gerar a frase.
# O retorno é a resposta textual; o pipeline guarda as evidências separadamente.
def responder_com_evidencias(
    pergunta: str,
    evidencias: list[Evidencia],
) -> str:
    # Transformamos objetos de evidência em texto numerado para o modelo citar.
    contexto = formatar_evidencias(
        evidencias
    )

    system = """
Responda SOMENTE com base nas evidências.

Regras:
- não use conhecimento externo;
- não invente;
- preserve números, condições e exceções;
- cite as evidências como [1], [2], etc.;
- se houver conflito, explicite;
- responda em português.
""".strip()

    return chamar_llm(
        system,
        (
            f"PERGUNTA:\n{pergunta}\n\n"
            f"EVIDÊNCIAS:\n{contexto}"
        ),
        max_new_tokens=320,
    )

# PARTE X — Pipeline completo

## 25. Execução ponta a ponta

Observe a ordem:

1. ambiguidade;
2. roteamento;
3. controle de permissão;
4. pré-processamento **se houver vetor**;
5. execução da(s) rota(s);
6. pós-processamento vetorial;
7. união das evidências;
8. avaliação de suficiência;
9. resposta;
10. log.

In [ ]:
# Objetivo: coordenar todas as etapas e devolver um registro completo da tentativa.
# A função é o ponto de entrada: recebe pergunta/perfil e chama funções menores.
# if/elif escolhe uma única rota; depois as rotas convergem na avaliação e na resposta.
# Guardar plano, consultas, evidências e latência permite investigar onde houve um problema.
# Perguntas ambíguas retornam antes da gravação do log; exceções também podem interromper
# a função antes dessa etapa. Os testes finais capturam erros de cada caso separadamente.

CAMINHO_LOG = Path(
    PASTA_RAIZ / "rag_hibrido_pre_pos_log.jsonl"
)


# Recebe a pergunta e coordena decisão, consulta, avaliação, resposta e log.
# As funções auxiliares isolam responsabilidades para facilitar testes e diagnóstico.
# O dicionário retornado funciona como um relatório da execução; o formato abreviado
# de 'esclarecer' exige que os consumidores verifiquem status antes de acessar outros campos.
def executar_pipeline(
    pergunta: str,
    perfil: str = "publico",
) -> dict:
    # perf_counter mede intervalos de tempo com boa resolução.
    # Guardamos o início e subtraímos do final para calcular a latência desta chamada.
    inicio_total = (
        time.perf_counter()
    )

    # ---------------------------------
    # 1. Ambiguidade
    # ---------------------------------

    ambiguidade = (
        detectar_ambiguidade_llm(
            pergunta
        )
    )

    if normalizar_bool(
        ambiguidade.get("ambigua"),
        padrao=False,
    ):
        return {
            "status": "esclarecer",
            "pergunta": pergunta,
            "ambiguidade": ambiguidade,
        }

    # ---------------------------------
    # 2. Roteamento
    # ---------------------------------

    plano = rotear_llm(
        pergunta
    )

    rota = plano[
        "rota"
    ]

    escopo = plano.get(
        "escopo"
    )

    # ---------------------------------
    # 3. Permissão
    # ---------------------------------

    validar_permissao(
        perfil,
        escopo,
    )

    evidencias = []
    # Criamos as mesmas chaves para todas as rotas; None indica etapa não utilizada.
    # Isso simplifica exibição/log e distingue 'não executado' de um texto vazio.
    detalhes = {
        "pre_processamento": None,
        "sql": None,
        "explain": None,
        "consulta_grafo": None,
        "ids_identificados": None,
        "ids_refinados": None,
        "estrategia_sql": None,
        "sql_params": None,
        "controle_acesso": "camada_de_dados",
    }

    # ---------------------------------
    # 4. ROTA VETOR
    # ---------------------------------

    # Cada ramo preenche evidencias no formato comum Evidencia.
    # Isso permite reaproveitar avaliação, resposta e log após a escolha da rota.
    if rota == "vetor":
        candidatos, pre = (
            recuperar_multi_query(
                pergunta,
                escopo=escopo,
                perfil=perfil,
            )
        )

        detalhes[
            "pre_processamento"
        ] = pre

        evidencias = (
            pos_processar_vetor(
                pergunta,
                candidatos,
            )
        )

    # ---------------------------------
    # 5. ROTA SQL
    # ---------------------------------

    elif rota == "sql":
        consulta_sql = nl2sql(
            pergunta
        )

        validar_intencao_sql(
            pergunta,
            consulta_sql,
        )

        resultado_sql, ev_sql = (
            executar_sql(
                consulta_sql,
                perfil=perfil,
            )
        )

        if not ev_sql:
            raise ValueError(
                "O SQL foi executado, mas não retornou evidências."
            )

        detalhes["sql"] = consulta_sql.sql
        detalhes["sql_params"] = list(
            consulta_sql.params
        )
        detalhes["estrategia_sql"] = (
            consulta_sql.origem
        )
        detalhes["explain"] = (
            resultado_sql["explain"]
        )

        evidencias = ev_sql

    # ---------------------------------
    # 6. ROTA GRAFO
    # ---------------------------------

    elif rota == "grafo":
        consulta = nl2graph(
            pergunta
        )

        detalhes[
            "consulta_grafo"
        ] = consulta

        evidencias = (
            executar_grafo(
                consulta,
                perfil=perfil,
            )
        )

    # ---------------------------------
    # 7. VETOR + SQL
    # ---------------------------------

    elif rota == "vetor_sql":
        candidatos, pre = (
            recuperar_multi_query(
                pergunta,
                escopo=escopo,
                perfil=perfil,
            )
        )

        detalhes[
            "pre_processamento"
        ] = pre

        ev_vetor = (
            pos_processar_vetor(
                pergunta,
                candidatos,
            )
        )

        ids = identificar_ids_llm(
            pergunta,
            ev_vetor,
        )

        detalhes[
            "ids_identificados"
        ] = ids

        if not ids:
            raise ValueError(
                "A busca vetorial recuperou evidências, "
                "mas nenhuma entidade estruturada foi resolvida."
            )

        consulta_sql, ids_finais = nl2sql_com_ids(
            pergunta,
            ids,
            perfil=perfil,
        )

        detalhes[
            "ids_refinados"
        ] = ids_finais

        try:
            validar_sql_respeita_ids(
                consulta_sql,
                ids_finais,
            )

            validar_intencao_sql(
                pergunta,
                consulta_sql,
                ids_resolvidos=ids_finais,
            )

            resultado_sql, ev_sql = (
                executar_sql(
                    consulta_sql,
                    perfil=perfil,
                )
            )

        # O reparo tenta uma nova geração apenas para SQL que veio do LLM.
        # Erros em templates são propagados para que um defeito de código seja investigado.
        # A consulta reparada passa novamente pelas validações antes de executar.
        except Exception as erro_sql:
            if consulta_sql.origem == "template_deterministico":
                raise RuntimeError(
                    "Falha em um template determinístico. "
                    "Não vamos mascarar erro de código com reparo por LLM."
                ) from erro_sql

            # Apenas SQL gerado pelo LLM recebe uma tentativa de reparo.
            system_reparo = """
Corrija a consulta SQLite.

Regras:
- retorne APENAS um SELECT;
- use somente o schema fornecido;
- use obrigatoriamente os IDs resolvidos fornecidos;
- não invente colunas;
- sem Markdown e sem explicações.
""".strip()

            sql_reparado = limpar_bloco(
                chamar_llm(
                    system_reparo,
                    f"""
SCHEMA:
{obter_schema_sql()}

PERGUNTA:
{pergunta}

IDs RESOLVIDOS:
{ids_finais}

SQL ANTERIOR:
{consulta_sql.sql}

ERRO:
{erro_sql}

SQL CORRIGIDO:
""".strip(),
                    max_new_tokens=180,
                )
            )

            consulta_sql = ConsultaSQL(
                sql=sql_reparado,
                params=(),
                origem="llm_reparo",
            )

            validar_sql_respeita_ids(
                consulta_sql,
                ids_finais,
            )

            validar_intencao_sql(
                pergunta,
                consulta_sql,
                ids_resolvidos=ids_finais,
            )

            resultado_sql, ev_sql = (
                executar_sql(
                    consulta_sql,
                    perfil=perfil,
                )
            )

        detalhes["sql"] = consulta_sql.sql
        detalhes["sql_params"] = list(
            consulta_sql.params
        )
        detalhes["estrategia_sql"] = (
            consulta_sql.origem
        )
        detalhes["explain"] = (
            resultado_sql["explain"]
        )

        # + concatena duas listas: mantemos a justificativa textual da entidade e
        # o resultado numérico. A avaliação híbrida verificará a presença dos dois tipos.
        evidencias = (
            ev_vetor + ev_sql
        )

    # ---------------------------------
    # 8. VETOR + GRAFO
    # ---------------------------------

    elif rota == "vetor_grafo":
        candidatos, pre = (
            recuperar_multi_query(
                pergunta,
                escopo=escopo,
                perfil=perfil,
            )
        )

        detalhes[
            "pre_processamento"
        ] = pre

        ev_vetor = (
            pos_processar_vetor(
                pergunta,
                candidatos,
            )
        )

        consulta = nl2graph(
            pergunta
        )

        detalhes[
            "consulta_grafo"
        ] = consulta

        ev_grafo = executar_grafo(
            consulta,
            perfil=perfil,
        )

        evidencias = (
            ev_vetor + ev_grafo
        )

    else:
        raise ValueError(
            f"Rota não implementada: {rota}"
        )

    # ---------------------------------
    # 9. Contexto suficiente?
    # ---------------------------------

    suficiencia = (
        avaliar_suficiencia(
            pergunta,
            evidencias,
            rota=rota,
        )
    )

    if not normalizar_bool(
        suficiencia.get("suficiente"),
        padrao=False,
    ):
        resposta = (
            "Não há evidência suficiente "
            "para responder com segurança."
        )
        status = "abster"

    else:
        resposta = (
            responder_com_evidencias(
                pergunta,
                evidencias,
            )
        )
        status = "responder"

    # ---------------------------------
    # 10. Log
    # ---------------------------------

    registro = {
        "status": status,
        "pergunta": pergunta,
        "perfil": perfil,
        "ambiguidade": ambiguidade,
        "plano": plano,
        "detalhes": detalhes,
        "evidencias": [
            # asdict transforma a dataclass em dicionário comum.
            # Isso prepara os campos da evidência para serialização em JSON e para o relatório.
            asdict(e)
            for e in evidencias
        ],
        "suficiencia": suficiencia,
        "resposta": resposta,
        "latencia_s": round(
            (
                time.perf_counter()
                - inicio_total
            ),
            3,
        ),
    }

    # with fecha o arquivo automaticamente ao sair do bloco, mesmo se ocorrer erro.
    # Modo 'a' acrescenta registros ao final. Cada json.dumps mais '\n' forma uma linha JSONL.
    # default=str converte valores não serializáveis em texto, perdendo seu tipo original.
    with CAMINHO_LOG.open(
        "a",
        encoding="utf-8",
    ) as arquivo:
        arquivo.write(
            json.dumps(
                registro,
                ensure_ascii=False,
                default=str,
            )
            + "\n"
        )

    return registro

## 26. Função de exibição

In [ ]:
# Objetivo: transformar o registro do pipeline em uma saída legível no notebook.
# A função recebe dados já calculados: imprimir novamente não faz uma nova busca.
# Campos opcionais são mostrados apenas quando a rota os produziu.
# O conteúdo de cada evidência é abreviado na tela; o registro mantém o valor completo.
# Retornar cedo no caso 'esclarecer' evita acessar campos que não foram produzidos.

# Exibe o dicionário de resultado em blocos legíveis para facilitar a inspeção.
# Imprime partes do registro em uma ordem útil para revisão.
# Usamos .get nos campos opcionais para lidar com diferenças entre rotas.
# O retorno None indica que esta função só apresenta dados, sem produzir uma nova resposta.
def mostrar(
    registro: dict,
) -> None:
    print("=" * 70)
    print(
        "STATUS:",
        registro.get("status"),
    )

    # Exibimos o esclarecimento já solicitado pelo pipeline.
    if registro.get(
        "status"
    ) == "esclarecer":
        print(
            registro["ambiguidade"].get(
                "pergunta_esclarecimento"
            )
        )
        return

    print("\nPLANO:")
    print(
        json.dumps(
            registro["plano"],
            ensure_ascii=False,
            indent=2,
        )
    )

    detalhes = registro.get(
        "detalhes",
        {}
    )

    # SQL, pré-processamento e grafo são opcionais: cada rota preenche
    # apenas os detalhes que utilizou.
    if detalhes.get(
        "pre_processamento"
    ):
        print("\nPRÉ-PROCESSAMENTO:")
        print(
            json.dumps(
                detalhes[
                    "pre_processamento"
                ],
                ensure_ascii=False,
                indent=2,
            )
        )

    if detalhes.get(
        "sql"
    ):
        print("\nSQL:")
        print(
            detalhes["sql"]
        )

        print(
            "\nESTRATÉGIA SQL:",
            detalhes.get(
                "estrategia_sql"
            ),
        )

        if detalhes.get(
            "sql_params"
        ):
            print(
                "PARÂMETROS SQL:",
                detalhes["sql_params"],
            )

        print("\nEXPLAIN:")
        print(
            detalhes["explain"]
        )

    if detalhes.get(
        "consulta_grafo"
    ):
        print("\nCONSULTA DE GRAFO:")
        print(
            json.dumps(
                detalhes[
                    "consulta_grafo"
                ],
                ensure_ascii=False,
                indent=2,
            )
        )

    print("\nSUFICIÊNCIA:")
    print(
        json.dumps(
            registro[
                "suficiencia"
            ],
            ensure_ascii=False,
            indent=2,
        )
    )

    print("\nEVIDÊNCIAS:")
    for indice, evidencia in enumerate(
        registro["evidencias"],
        start=1,
    ):
        print(
            f"\n[{indice}]",
            evidencia["tipo"],
            "|",
            evidencia["fonte"],
        )
        print(
            evidencia["conteudo"][:500]
        )

    print("\nRESPOSTA:")
    print(
        registro["resposta"]
    )

    print(
        "\nLATÊNCIA:",
        registro["latencia_s"],
        "s",
    )

## Diagnóstico rápido do detector de ambiguidade

Antes dos experimentos. Apenas a primeira pergunta deve resultar em `ambigua=True`.

In [ ]:
# Objetivo: comparar perguntas vagas com perguntas que identificam o alvo.
# O loop usa r como nome curto para o dicionário retornado pelo detector.
# Imprimimos o tipo do campo ambigua para conferir se ele é bool.
# Isso importa porque o texto "false" é não vazio e seria verdadeiro em um if comum.

perguntas_diagnostico = [
    "Qual é a carga horária?",
    "Qual é a carga horária da CI1218?",
    "Qual é a carga horária total dos dois TCCs?",
    "Quais são os pré-requisitos da CI1218?",
    "O que Bancos de Dados aborda sobre processamento de consultas?",
    "Qual é a carga horária da disciplina que trata de processamento de consultas e otimização?",
]

for p in perguntas_diagnostico:
    r = detectar_ambiguidade_llm(p)
    print("\nPERGUNTA:", p)
    print("ambigua =", r.get("ambigua"), "| tipo =", type(r.get("ambigua")).__name__)
    print("motivo =", r.get("motivo"))

# PARTE XI — Experimentos

### Diagnóstico de suficiência antes do Experimento A

Esta célula mostra os termos centrais encontrados nas evidências. Para a pergunta sobre processamento de consultas, a cobertura deve ser suficiente para evitar uma abstenção indevida.


> A função atual de suficiência é `avaliar_suficiencia()` e recebe a rota explicitamente. Para este teste, usamos `rota="vetor"`.

In [ ]:
# Objetivo: localizar problemas de recuperação antes de gerar uma resposta.
# Separamos busca, pós-processamento e avaliação de suficiência em três chamadas.
# Assim podemos comparar os trechos com o motivo da decisão e seus scores.
# Atenção à sintaxe: strings adjacentes são coladas sem acrescentar espaços.
# No texto abaixo, 'disciplina' e 'aborda' ficam unidos; observe isso ao interpretar o teste.

pergunta_teste = (
    "Qual disciplina"
    "aborda sobre processamento de consultas?"
)

candidatos_teste, pre_teste = recuperar_multi_query(
    pergunta_teste,
    escopo="disciplina",
)

evidencias_teste = pos_processar_vetor(
    pergunta_teste,
    candidatos_teste,
)

avaliacao_teste = avaliar_suficiencia(
    pergunta_teste,
    evidencias_teste,
    rota="vetor",
)

print(
    json.dumps(
        avaliacao_teste,
        ensure_ascii=False,
        indent=2,
    )
)

print("\nEVIDÊNCIAS:")
for e in evidencias_teste:
    print("-", e.fonte)
    print(" ", e.conteudo[:350])

## 27. Experimento A — vetor

Observe:
- transformações da consulta;
- múltiplas buscas;
- RRF;
- filtro;
- deduplicação;
- reranking;
- compressão.

In [ ]:
# Objetivo: executar o percurso completo de uma pergunta sobre conteúdo.
# O Python calcula executar_pipeline(...) primeiro e passa seu retorno a mostrar(...).
# Espera-se a rota vetor porque a pergunta pede explicação textual, não um cálculo.
# Confira os trechos citados para avaliar se sustentam a resposta produzida.

mostrar(
    executar_pipeline(
        "O que a disciplina de Bancos de Dados "
        "aborda sobre processamento de consultas?"
    )
)

### Diagnóstico: TCC no SQL

Aqui verificamos separadamente intenção -> SQL -> resultado. O SQL deve usar `SUM` e considerar `TCC1` e `TCC2`.

In [ ]:
# Objetivo: separar a geração do SQL, sua validação e o cálculo do total dos TCCs.
# validar_intencao_sql exige uma operação de soma para essa pergunta.
# Depois da execução, inspecione se os dois IDs foram considerados e compare as linhas
# retornadas com os valores cadastrados. Um SELECT válido pode ainda responder outra pergunta.

pergunta_tcc = (
    "Qual é a carga horária total "
    "do TCC1 e do TCC2?"
)

sql_tcc = nl2sql(
    pergunta_tcc
)

print("SQL:")
print(sql_tcc)

validar_intencao_sql(
    pergunta_tcc,
    sql_tcc,
)

resultado_tcc, evidencias_tcc = executar_sql(
    sql_tcc
)

print("\nRESULTADO:")
print(resultado_tcc["linhas"])

print("\nEVIDÊNCIAS:")
for e in evidencias_tcc:
    print("-", e.conteudo)

## 28. Experimento B — NL2SQL

Pergunta com agregação.

In [ ]:
# Objetivo: executar a soma dos TCCs pelo pipeline completo.
# Ao contrário do diagnóstico isolado, aqui também são feitos roteamento,
# avaliação de suficiência, geração da resposta e gravação do registro.
# Observe a consulta SQL além da frase final: é ela que deve realizar a soma.

mostrar(
    executar_pipeline(
        #"Qual é a carga horária total dos do TCC?"
        "Qual é a carga horária "
        "total do TCC1 e do TCC2?"
    )
)

### Diagnóstico: consulta do grafo

Aqui verificamos linguagem natural -> consulta estruturada -> relações encontradas.

In [ ]:
# Objetivo: verificar a consulta gerada e as relações encontradas no grafo.
# Guardar a consulta em uma variável permite imprimir seus campos antes da execução.
# Isso ajuda a distinguir uma direção incorreta de uma relação ausente nos dados.
# As evidências impressas são caminhos cadastrados; ainda não são uma resposta do LLM.

pergunta_grafo = (
    "Quais são os pré-requisitos da CI1218?"
)

consulta_grafo_teste = nl2graph(
    pergunta_grafo
)

print("CONSULTA:")
print(
    json.dumps(
        consulta_grafo_teste,
        ensure_ascii=False,
        indent=2,
    )
)

evidencias_grafo_teste = executar_grafo(
    consulta_grafo_teste
)

print("\nRELAÇÕES:")
for e in evidencias_grafo_teste:
    print("-", e.conteudo)

## 29. Experimento C — NL2Graph

Pergunta relacional.

O objetivo é mostrar que similaridade semântica não substitui uma relação explícita.

In [ ]:
# Objetivo: testar a rota de grafo dentro do pipeline completo.
# Compare as relações exibidas com o diagnóstico isolado anterior.
# A frase final deve resumir esses caminhos, preservando a diferença entre relações
# diretas e indiretas definida na consulta.

mostrar(
    executar_pipeline(
        "Quais são os pré-requisitos da CI1218?"
    )
)

### Diagnóstico da rota vetor + SQL

Este teste separa as três etapas para mostrar exatamente onde uma falha ocorre: (1) recuperação vetorial, (2) resolução da entidade, (3) NL2SQL.

In [ ]:
# Objetivo: testar somente a passagem de IDs conhecidos para a consulta SQL.
# Fornecemos ['CI1218'] manualmente para isolar essa etapa da busca vetorial.
# Portanto, um resultado correto aqui não prova que a busca descobriu a entidade.
# origem mostra template ou LLM; params mostra os valores separados do texto SQL.

consulta_teste_id, ids_teste_finais = nl2sql_com_ids(
    "Qual é a carga horária da disciplina que trata de processamento de consultas e otimização?",
    ["CI1218"],
)

print("IDs finais:", ids_teste_finais)
print("origem:", consulta_teste_id.origem)
print("params:", consulta_teste_id.params)
print(consulta_teste_id.sql)

In [ ]:
# Objetivo: acompanhar a rota híbrida passo a passo para localizar eventuais falhas.
# Primeiro buscamos texto; depois resolvemos IDs; finalmente consultamos valores exatos.
# As variáveis intermediárias permitem inspecionar a saída de cada etapa.
# O if impede gerar SQL quando nenhuma entidade foi identificada nas evidências.

pergunta_vsql = (
    "Qual é a carga horária da disciplina "
    "que trata de processamento de consultas "
    "e otimização?"
)

candidatos_vsql, pre_vsql = recuperar_multi_query(
    pergunta_vsql,
    escopo="disciplina",
)

ev_vsql = pos_processar_vetor(
    pergunta_vsql,
    candidatos_vsql,
)

print("1. EVIDÊNCIAS VETORIAIS")
for e in ev_vsql:
    print("-", e.fonte)
    print(" ", e.conteudo[:250])

ids_vsql = identificar_ids_llm(
    pergunta_vsql,
    ev_vsql,
)

print("\n2. IDs RESOLVIDOS")
print(ids_vsql)

# Só continuamos quando existe pelo menos um candidato.
# Essa divisão ajuda a descobrir se a falha veio da busca, da identificação ou do SQL.
if ids_vsql:
    consulta_vsql, ids_finais_vsql = nl2sql_com_ids(
        pergunta_vsql,
        ids_vsql,
    )

    print("\nIDs FINAIS APÓS REFINAMENTO")
    print(ids_finais_vsql)

    print("\n3. SQL GERADO")
    print("origem:", consulta_vsql.origem)
    print("params:", consulta_vsql.params)
    print(consulta_vsql.sql)

    validar_sql_respeita_ids(
        consulta_vsql,
        ids_finais_vsql,
    )

    resultado_vsql, ev_sql_vsql = executar_sql(
        consulta_vsql
    )

    print("\n4. RESULTADO SQL")
    print(resultado_vsql["linhas"])
else:
    print(
        "\nFalha localizada: a entidade não foi resolvida "
        "a partir das evidências vetoriais."
    )

## 30. Experimento D — vetor + SQL

Esta é a demonstração híbrida mais importante.

O usuário descreve semanticamente a disciplina; depois SQL recupera o valor exato.

In [ ]:
# Objetivo: testar a combinação de identificação semântica e consulta numérica.
# A pergunta descreve uma disciplina pelo conteúdo, sem fornecer seu código.
# Por isso esperamos vetor_sql: descobrir o ID e depois consultar sua carga horária.
# Confira IDs candidatos/refinados e estratégia SQL, além da resposta final.

mostrar(
    executar_pipeline(
        "Qual é a carga horária da disciplina "
        "que trata de processamento de consultas "
        "e otimização?"
    )
)

## 31. Experimento E — ambiguidade

Aqui o sistema deve preferir esclarecer em vez de recuperar.

In [ ]:
# Objetivo: observar o comportamento quando a pergunta não identifica o alvo.
# 'Qual é a carga horária?' pode se referir a várias entidades.
# O resultado esperado é 'esclarecer', com uma pergunta complementar.
# Esse retorno antecipado evita consultar dados de uma entidade escolhida por suposição.

mostrar(
    executar_pipeline(
        "Qual é a carga horária?"
    )
)

## 32. Experimento F — permissão simulada

A política é didática.

O perfil `somente_disciplina` não deve consultar informações de estágio.

In [ ]:
# Objetivo: demonstrar a recusa de uma consulta fora do escopo do perfil.
# Passamos perfil='somente_disciplina', mas perguntamos sobre estágio.
# except PermissionError captura especificamente a recusa esperada e a exibe.
# Outros tipos de erro não são capturados aqui, para não confundi-los com acesso negado.

try:
    mostrar(
        executar_pipeline(
            "O que o regulamento diz sobre estágio obrigatório?",
            perfil="somente_disciplina",
        )
    )
except PermissionError as erro:
    print(
        "ACESSO NEGADO:",
        erro,
    )

## 33. Experimento G — embeddings desatualizados

Para demonstrar invalidação:

```python
VERSOES_ATUAIS["estagio"] = "2025"
```

Os chunks com `embedding_version="2024"` serão removidos no pós-processamento.

In [ ]:
# Objetivo: simular um índice com versão diferente da configuração atual.
# A atribuição abaixo está comentada: executar esta célula não altera a configuração.
# Se ativada, o filtro passa a comparar os chunks de estágio com a versão '2025',
# mas isso não baixa um novo regulamento nem recalcula embeddings.
# Após o experimento, restaure '2024' para usar novamente o corpus disponível.

# Controle apenas se tivessemos outras versões.
# VERSOES_ATUAIS["estagio"] = "2025"

## 34. Pergunta livre

In [ ]:
# Objetivo: permitir uma pergunta digitada pelo aluno.
# input pausa a execução até receber texto; strip remove espaços nas extremidades.
# if pergunta só continua quando sobrou algum conteúdo depois dessa limpeza.
# Esta célula é opcional e interativa: pule-a ao executar apenas os testes da atividade.

pergunta = input(
    "Pergunta: "
).strip()

if pergunta:
    mostrar(
        executar_pipeline(
            pergunta
        )
    )

## 35. Inspecionar o último log

In [ ]:
# Objetivo: revisar o último registro persistido pelo pipeline.
# O formato JSONL guarda um objeto JSON por linha; splitlines separa esses registros.
# O índice -1 seleciona o último item da lista, e json.loads reconstrói o dicionário.
# Verificamos arquivo e conteúdo antes de indexar para evitar erros em listas vazias.
# O último log pode ser de uma execução anterior, pois nem toda tentativa chega a gravá-lo.

if CAMINHO_LOG.exists():
    linhas = (
        CAMINHO_LOG
        .read_text(
            encoding="utf-8"
        )
        .splitlines()
    )

    if linhas:
        ultimo = json.loads(
            linhas[-1]
        )

        print(
            json.dumps(
                ultimo,
                ensure_ascii=False,
                indent=2,
            )
        )
else:
    print(
        "Nenhum log foi criado."
    )

## 36. Atividade final — testes e relatório de entrega

Execute as células anteriores antes desta seção. Não é necessário executar a célula de **Pergunta livre** para produzir o relatório.

### O que será testado?

Cada pergunta da atividade possui uma **rota esperada**. O teste só é considerado aprovado quando três condições são verdadeiras: (1) o pipeline retorna o status `responder`, (2) escolhe a rota esperada e (3) recupera pelo menos uma evidência. Essas verificações não avaliam automaticamente se a resposta está correta ou se a evidência sustenta a conclusão. Use o campo `evidencia_esperada` e os trechos recuperados para fazer essa revisão manual.

A primeira célula executa os testes e guarda os resultados em memória. A segunda converte esses resultados em um arquivo Markdown, que pode ser aberto, revisado e entregue. Os comentários no código explicam o papel de cada bloco.

In [ ]:
# Objetivo: executar as cinco perguntas e reunir resultados comparáveis.
# Uma lista mantém a ordem dos casos; cada dicionário agrupa pergunta, rota e evidência esperada.
# Guardamos resultados e erros em memória para que o relatório use exatamente esta execução.
# 'APROVADO' significa apenas status responder, rota esperada e lista de evidências não vazia.
# O texto evidencia_esperada orienta a revisão humana; o teste não verifica seu conteúdo
# automaticamente nem certifica a correção factual da resposta.

from datetime import datetime

# Cada dicionário descreve UM teste. Manter essas informações juntas
# facilita acrescentar ou alterar perguntas no futuro.
CASOS_ATIVIDADE_FINAL = [
    {
        "pergunta": "O que CI1218 aborda sobre transações?",
        "rota_esperada": "vetor",
        "evidencia_esperada": "Trecho da ementa ou do programa da CI1218 sobre transações.",
    },
    {
        "pergunta": "Qual é a carga horária total do TCC1 e do TCC2?",
        "rota_esperada": "sql",
        "evidencia_esperada": "Registros estruturados de TCC1 e TCC2 e a soma das cargas horárias.",
    },
    {
        "pergunta": "Quais são os pré-requisitos da CI1218?",
        "rota_esperada": "grafo",
        "evidencia_esperada": "Relações de pré-requisito que chegam à entidade CI1218.",
    },
    {
        "pergunta": "Qual é a carga horária da disciplina que trata de processamento de consultas e otimização?",
        "rota_esperada": "vetor_sql",
        "evidencia_esperada": "Trecho que identifica semanticamente a disciplina e consulta SQL com a carga horária.",
    },
    {
        "pergunta": "Quando o aluno pode realizar estágio obrigatório?",
        "rota_esperada": "vetor",
        "evidencia_esperada": "Trecho do regulamento ou PPC que estabelece o requisito para cursar o estágio.",
    },
]


# Cria uma prévia legível para tabelas e listas do relatório.
# str converte o valor em texto e [:limite] seleciona seus primeiros caracteres.
# O caractere de reticências avisa que houve corte; a prévia não substitui o registro completo.
def resumir_texto(valor, limite=360):
    """Converte qualquer valor em uma frase curta para caber no relatório."""

    # As respostas podem conter quebras de linha; para uma tabela Markdown
    # elas são trocadas por espaços.
    texto = str(valor or "").replace("\n", " ").strip()

    # O corte evita que uma evidência muito longa deixe o relatório difícil de ler.
    return texto[:limite] + ("…" if len(texto) > limite else "")


# Reúne nomes de fontes sem repetir itens idênticos.
# O registro serializado contém dicionários: usamos e['fonte']/.get em vez de e.fonte.
# O campo pagina é procurado no nível principal; na estrutura atual ele pode estar
# nos metadados, enquanto a fonte vetorial já inclui a página em seu próprio texto.
def fontes_evidencias(registro):
    """Extrai uma lista sem repetições das fontes usadas pelo pipeline."""

    fontes = []

    # `evidencias` é a lista devolvida por executar_pipeline. Cada item
    # informa de onde veio o dado: PDF, banco SQLite ou grafo.
    for evidencia in registro.get("evidencias", []):
        fonte = evidencia.get("fonte") or evidencia.get("documento")
        pagina = evidencia.get("pagina")
        item = str(fonte or "Fonte não identificada")
        if pagina is not None:
            item += f" (p. {pagina})"

        # A mesma fonte pode aparecer em vários trechos. Ela entra uma vez
        # na lista para o relatório permanecer claro.
        if item not in fontes:
            fontes.append(item)
    return fontes


# Combina uma prévia do conteúdo com sua fonte para revisão humana.
# if conteudo evita incluir linhas vazias quando faltou texto na evidência.
# O corte pode omitir condições; consulte o registro completo em caso de dúvida.
def trechos_evidencias(registro, limite=280):
    """Monta pequenos trechos que permitem conferir a resposta manualmente."""

    trechos = []
    for evidencia in registro.get("evidencias", []):
        fonte = evidencia.get("fonte") or "Fonte não identificada"
        conteudo = resumir_texto(evidencia.get("conteudo"), limite)
        if conteudo:
            trechos.append(f"{fonte}: {conteudo}")
    return trechos


# Executa cada caso e acumula sucesso ou erro sem encerrar a bateria inteira.
# O padrão casos=CASOS_ATIVIDADE_FINAL é associado ao definir a função.
# Se substituir a lista global depois, passe a nova lista explicitamente ou redefina a função.
# O critério automático mede estrutura da execução; a evidência esperada é verificada manualmente.
def executar_testes_atividade(casos=CASOS_ATIVIDADE_FINAL):
    """Executa todos os casos e devolve resultados prontos para o relatório."""

    resultados = []

    # enumerate cria uma numeração começando em 1 apenas para facilitar
    # a leitura do andamento no notebook.
    for numero, caso in enumerate(casos, start=1):
        print(f"[{numero}/{len(casos)}] {caso['pergunta']}")
        try:
            # Esta é a chamada principal: ela roteia a pergunta, recupera
            # evidências e produz a resposta final.
            registro = executar_pipeline(caso["pergunta"])

            # Usamos .get para evitar um erro caso algum campo não exista.
            rota_obtida = registro.get("plano", {}).get("rota")
            # bool(lista) é False para lista vazia e True para lista com itens.
            # Isso testa presença, não relevância ou correção do conteúdo recuperado.
            tem_evidencia = bool(registro.get("evidencias"))
            passou = (
                # As três regras de aprovação tornam o teste auditável.
                registro.get("status") == "responder"
                and rota_obtida == caso["rota_esperada"]
                and tem_evidencia
            )
            resultado = {
                # **caso copia os campos de entrada para o resultado.
                # Juntar esperado e obtido no mesmo dicionário facilita comparar e gerar o relatório.
                **caso,
                "status": registro.get("status"),
                "rota_obtida": rota_obtida,
                "tem_evidencia": tem_evidencia,
                "passou": passou,
                "resposta": registro.get("resposta", ""),
                "fontes": fontes_evidencias(registro),
                "trechos": trechos_evidencias(registro),
                "registro": registro,
                "erro": None,
            }
        except Exception as erro:
            # Um erro em uma pergunta não interrompe as demais. Em vez disso,
            # ele é registrado para aparecer claramente no relatório.
            resultado = {
                **caso,
                "status": "erro",
                "rota_obtida": None,
                "tem_evidencia": False,
                "passou": False,
                "resposta": "",
                "fontes": [],
                "trechos": [],
                "registro": None,
                "erro": f"{type(erro).__name__}: {erro}",
            }

        resultados.append(resultado)

        # Feedback imediato: útil para descobrir qual caso precisa de revisão.
        marcador = "APROVADO" if resultado["passou"] else "REVISAR"
        print(f"  {marcador} | rota: {resultado['rota_obtida']} | status: {resultado['status']}")

    return resultados


# Guardamos a lista em uma variável para que a próxima célula possa usá-la
# sem repetir consultas ao modelo e às fontes de dados.
RESULTADOS_ATIVIDADE_FINAL = executar_testes_atividade()
# True conta como 1 e False como 0 em uma soma no Python.
# O gerador fornece um booleano por teste, permitindo contar aprovações sem outro laço.
aprovados = sum(item["passou"] for item in RESULTADOS_ATIVIDADE_FINAL)
print(f"\nResumo: {aprovados}/{len(RESULTADOS_ATIVIDADE_FINAL)} testes aprovados.")

In [ ]:
# Objetivo: gerar um arquivo de entrega com os resultados já obtidos.
# Montamos uma lista de linhas e juntamos tudo ao final, separando dados e apresentação.
# O Markdown pode ser aberto em editores de texto e revisado antes da entrega.
# Respostas/trechos são abreviados; os registros completos permanecem na variável dos testes.
# A gravação usa sempre o mesmo nome e substitui um relatório anterior nesse caminho.
# Revise fontes, números e casos marcados para revisão antes de entregar o arquivo.

# Faz o caractere | aparecer como texto dentro da tabela.
# Sem a barra invertida, Markdown interpretaria | como separação de colunas.
# Essa função trata esse delimitador; não é um sanitizador completo de Markdown.
def escapar_tabela_markdown(texto):
    """Prepara um texto para uso seguro dentro de uma célula de tabela."""

    # Em Markdown, | separa colunas. A barra invertida faz o caractere
    # aparecer como texto comum, sem quebrar a tabela.
    return resumir_texto(texto, limite=220).replace("|", "\\|")


# Converte os resultados dos testes em um arquivo Markdown.
# Recebe resultados, não perguntas, para não executar o modelo ao refazer a apresentação.
# O argumento padrão aponta para a lista existente na definição; após novos testes,
# passe a lista atual explicitamente ou reexecute esta célula que define a função.
def gerar_relatorio_atividade(resultados=RESULTADOS_ATIVIDADE_FINAL):
    """Cria o relatório Markdown a partir dos resultados já calculados."""

    # Primeiro calculamos um resumo numérico para apresentar logo no início.
    total = len(resultados)
    aprovados = sum(item["passou"] for item in resultados)
    linhas = [
        "# Relatório de entrega — Atividade final: RAG híbrido",
        "",
        f"Gerado em: {datetime.now().strftime('%d/%m/%Y %H:%M')}",
        "",
        "## Objetivo",
        "",
        "Validar a seleção de rota e a recuperação de evidências para as perguntas propostas na atividade final.",
        "",
        "## Resultado dos testes",
        "",
        f"**{aprovados}/{total}** testes aprovados. Um teste é aprovado quando a resposta é gerada, a rota coincide com a esperada e há ao menos uma evidência recuperada.",
        "",
        "| # | Pergunta | Rota esperada | Rota obtida | Status | Evidência | Resultado |",
        "|---:|---|---|---|---|---|---|",
    ]

    for numero, item in enumerate(resultados, start=1):
        # Esta tabela dá uma visão rápida de todos os testes.
        resultado = "Aprovado" if item["passou"] else "Revisar"
        linhas.append(
            f"| {numero} | {escapar_tabela_markdown(item['pergunta'])} | "
            f"{item['rota_esperada']} | {item['rota_obtida'] or '-'} | "
            f"{item['status']} | {'Sim' if item['tem_evidencia'] else 'Não'} | {resultado} |"
        )

    # append adiciona um item; extend adiciona cada item de uma sequência.
    # As strings vazias produzem linhas em branco, necessárias para separar blocos Markdown.
    linhas.extend(["", "## Evidências e respostas", ""])
    for numero, item in enumerate(resultados, start=1):
        # Depois da visão geral, registramos o detalhe necessário para
        # conferir a origem de cada resposta.
        linhas.extend([
            f"### {numero}. {item['pergunta']}",
            f"- Rota esperada: `{item['rota_esperada']}`",
            f"- Rota obtida: `{item['rota_obtida'] or '-'}`",
            f"- Evidência esperada: {item['evidencia_esperada']}",
            f"- Fontes recuperadas: {', '.join(item['fontes']) or 'Nenhuma'}",
            f"- Resposta do sistema: {resumir_texto(item['resposta']) or 'Não gerada'}",
        ])
        for trecho in item["trechos"]:
            linhas.append(f"- Evidência recuperada: {trecho}")
        if item["erro"]:
            linhas.append(f"- Erro: `{item['erro']}`")
        linhas.append("")

    linhas.extend([
        "## Conclusão",
        "",
        "O relatório registra a rota utilizada, a evidência recuperada e a resposta produzida para permitir auditoria e revisão dos casos que não passaram.",
    ])

    caminho = PASTA_RAIZ / "relatorio_atividade_final.md"

    # write_text cria (ou atualiza) somente o relatório desta atividade.
    # UTF-8 preserva corretamente os acentos em português.
    # '\n'.join(linhas) intercala quebras de linha entre os itens.
    # write_text substitui o conteúdo do arquivo. Para guardar execuções diferentes,
    # seria necessário usar nomes distintos ou copiar o relatório antes de executar de novo.
    caminho.write_text("\n".join(linhas) + "\n", encoding="utf-8")
    return caminho


# A função devolve o caminho para que seja fácil localizar o arquivo criado.
CAMINHO_RELATORIO_ATIVIDADE = gerar_relatorio_atividade()
print(f"Relatório gerado em: {CAMINHO_RELATORIO_ATIVIDADE.resolve()}")